In [ ]:
# Stage 1 - 3DYoga90 skeleton parquet ETL into (T, 33, 4) clips + metadata.

import os, io, json, re, zipfile, csv
import numpy as np
import pandas as pd

# Paths
JSON_PATH   = "path/to/3DYoga90.json"
PARQUET_DIR = "path/to/3dyoga90-skeletons"
OUT_ROOT    = "path/to/working/extracted_3dyoga90"
MIN_FRAMES  = 16

# 3DYoga90 label -> 7-target canonical name (warrior-1 only;
# extended-triangle is the only triangle class in 3DYoga90).
TARGET_MAP = {
    "downward-dog":      "downward_facing_dog",
    "tree":              "tree",
    "extended-triangle": "triangle",
    "warrior-1":         "warrior",
    "mountain":          "mountain",
    "cobra":             "cobra",
    "corpse":            "corpse",
}

os.makedirs(f"{OUT_ROOT}/clips", exist_ok=True)
N_POSE_LANDMARKS = 33

# 1. Load + filter 3DYoga90.json
with open(JSON_PATH, "r") as f:
    groups = json.load(f)

filtered = []
for g in groups:
    target = TARGET_MAP.get(g["pose"])
    if target is None:
        continue
    for inst in g["instances"]:
        filtered.append({**inst, "pose": target, "original_label": g["pose"]})

print(f"Matched {len(filtered)} sequences across 7 target poses")
counts = pd.Series([r["pose"] for r in filtered]).value_counts().sort_index()
for t, n in counts.items():
    print(f"  {t:25s} {n:4d}")

# 2. Index parquets by sequence_id
parquet_index = {
    os.path.splitext(os.path.basename(f))[0]: os.path.join(PARQUET_DIR, f)
    for f in os.listdir(PARQUET_DIR)
    if f.lower().endswith(".parquet")
}
print(f"\nIndexed {len(parquet_index)} parquets in the directory")
if not parquet_index:
    raise RuntimeError(
        "No parquets found. Check PARQUET_DIR path and contents; "
        "if filenames aren't <sequence_id>.parquet, adjust the stem extraction."
    )

# 3. ISLR parquet -> (T, 33, 4) clip
def parquet_file_to_clip(file_path):
    try:
        df = pd.read_parquet(
            file_path,
            columns=["frame", "type", "landmark_index", "x", "y", "z"],
        )
    except Exception:
        return None, {}

    pose_df = df[df["type"] == "pose"]
    if pose_df.empty:
        return None, {}

    frames = sorted(pose_df["frame"].unique())
    T = len(frames)
    frame_to_t = {f: t for t, f in enumerate(frames)}

    clip = np.full((T, N_POSE_LANDMARKS, 3), np.nan, dtype=np.float32)
    t_idx = pose_df["frame"].map(frame_to_t).to_numpy()
    l_idx = pose_df["landmark_index"].to_numpy()
    clip[t_idx, l_idx, 0] = pose_df["x"].to_numpy()
    clip[t_idx, l_idx, 1] = pose_df["y"].to_numpy()
    clip[t_idx, l_idx, 2] = pose_df["z"].to_numpy()

    # Synthesize visibility from NaN-presence (ISLR has no vis scores).
    detected = ~np.isnan(clip).any(axis=2)
    vis      = detected.astype(np.float32)[..., None]
    clip     = np.nan_to_num(clip, nan=0.0)
    clip     = np.concatenate([clip, vis], axis=2).astype(np.float32)

    frames_detected = int(detected.any(axis=1).sum())
    return clip, {
        "frames_kept":     T,
        "frames_detected": frames_detected,
        "detection_rate":  round(frames_detected / max(T, 1), 4),
    }

# 4. Main ETL loop
metadata_rows = []
clip_counter  = 0
skipped_missing = skipped_empty = skipped_short = 0

for rec in filtered:
    seq_id   = str(rec["sequence_id"])
    parquet_path = parquet_index.get(seq_id)
    if parquet_path is None:
        skipped_missing += 1
        continue

    clip, stats = parquet_file_to_clip(parquet_path)
    if clip is None:
        skipped_empty += 1
        continue
    if stats["frames_kept"] < MIN_FRAMES:
        skipped_short += 1
        continue

    safe_pose = rec["pose"]
    out_name  = f"{safe_pose}__correct__{clip_counter:04d}.npy"
    out_path  = os.path.join(OUT_ROOT, "clips", out_name)
    np.save(out_path, clip)

    metadata_rows.append({
        "clip_index":      clip_counter,
        "pose":            rec["pose"],
        "original_label":  rec["original_label"],
        "quality_label":   "correct",  # all 3DYoga90 is correct
        "frames_kept":     stats["frames_kept"],
        "frames_detected": stats["frames_detected"],
        "detection_rate":  stats["detection_rate"],
        "clip_path":       out_path,
        "source_file":     parquet_path,
        "split":           rec.get("split", ""),
        "dataset":         "3dyoga90",
    })
    clip_counter += 1
    if clip_counter % 100 == 0:
        print(f"  processed {clip_counter} / {len(filtered)} clips...")

# 5. Write metadata.csv
import csv

fieldnames = ["clip_index", "pose", "original_label", "quality_label",
              "frames_kept", "frames_detected", "detection_rate",
              "clip_path", "source_file", "split", "dataset"]
meta_path = f"{OUT_ROOT}/metadata.csv"
with open(meta_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(metadata_rows)

# 6. Summary
print("\n" + "=" * 55)
print(f"Stage 1 complete - 3DYoga90 ETL")
print(f"  clips saved       : {clip_counter}")
print(f"  skipped (missing) : {skipped_missing}   seq_id not found in parquet files")
print(f"  skipped (empty)   : {skipped_empty}     no pose rows in parquet")
print(f"  skipped (short)   : {skipped_short}     < {MIN_FRAMES} frames")
print(f"  metadata          : {meta_path}")

if metadata_rows:
    meta = pd.DataFrame(metadata_rows)
    print("\nClips per target pose:")
    print(meta.groupby("pose")["clip_index"].count().to_string())
    print("\nMean detection rate by pose:")
    print(meta.groupby("pose")["detection_rate"].mean().map("{:.2%}".format).to_string())
    print("\nFrames per clip:")
    print(meta["frames_kept"].describe().round(1).to_string())
    print("\nOriginal 3DYoga90 labels included (audit):")
    print(meta["original_label"].value_counts().to_string())


In [ ]:
# Stage 2 - feature engineering + synthetic-incorrect augmentation + windowing.

import os, numpy as np, pandas as pd

OUT = "path/to/working/windows"
os.makedirs(OUT, exist_ok=True)

# Windowing
WINDOW_SIZE    = 32        # halved from original 64
STRIDE         = 32        # non-overlapping
MIN_FRAMES     = 33        # drop clips with <= 32 frames
MIN_VIS_FRAC   = 0.6       # keep windows where >=60% of frames have any detection

# Augmentation: targeted angle corruption + global jitter.
N_CORRUPT_JOINTS_MIN = 1
N_CORRUPT_JOINTS_MAX = 2
CORRUPT_ANGLE_MIN_DEG = 0.0
CORRUPT_ANGLE_MAX_DEG = 45.0
JITTER_SIGMA = 0.01        # sigma in hip-centered, shoulder-normalized units

# 7 target poses, stable order (indices used in checkpoints below).
POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]
POSE_TO_ID = {p: i for i, p in enumerate(POSE_NAMES)}
N_POSES    = 7

# Quality: 0=correct (clean), 1=incorrect (synthetically corrupted).
LABEL_CORRECT, LABEL_INCORRECT = 0, 1

# 6 feedback joints (must match Stage 3 order).
JOINT_NAMES  = ["left_elbow", "right_elbow", "left_knee",
                "right_knee", "left_hip",    "right_hip"]
N_JOINTS     = 6

RNG = np.random.default_rng(42)
print(f"Window {WINDOW_SIZE} / stride {STRIDE} / min-frames {MIN_FRAMES}")
print(f"Poses ({N_POSES}): {POSE_NAMES}")


In [ ]:
# Landmark indices — BlazePose 33-point topology
LS, RS = 11, 12
LE, RE = 13, 14
LW, RW = 15, 16
LH, RH = 23, 24
LK, RK = 25, 26
LA, RA = 27, 28

# Downstream kinematic chains for chain rotation
# Elbow vertex -> rotate wrist + hand landmarks
# Knee vertex  -> rotate ankle + heel + foot_index
# Hip vertex   -> rotate entire leg chain (knee + ankle + heel + foot_index)
CHAIN_DOWNSTREAM = {
    "left_elbow":  [15, 17, 19, 21],          # wrist, pinky, index, thumb
    "right_elbow": [16, 18, 20, 22],
    "left_knee":   [27, 29, 31],              # ankle, heel, foot_index
    "right_knee":  [28, 30, 32],
    "left_hip":    [25, 27, 29, 31],          # knee + below
    "right_hip":   [26, 28, 30, 32],
}
# Vertex index (the joint we rotate around) for each of the 6 feedback joints
JOINT_VERTEX = {
    "left_elbow":  LE, "right_elbow": RE,
    "left_knee":   LK, "right_knee":  RK,
    "left_hip":    LH, "right_hip":   RH,
}
VIS_THRESH = 0.3
FEAT_DIM   = 210

def normalize_clip(clip):
    """Hip-center + shoulder-scale per frame. clip: (T, 33, 4)."""
    out = clip.copy()
    for t in range(len(out)):
        hip   = (out[t, LH, :3] + out[t, RH, :3]) / 2.0
        out[t, :, :3] -= hip
        scale = np.linalg.norm(out[t, LS, :3] - out[t, RS, :3]) + 1e-6
        out[t, :, :3] /= scale
    return out

def _angle_3pts_2d(a, b, c):
    ba = a - b; bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    return float(np.degrees(np.arccos(np.clip(np.dot(ba, bc) / denom, -1.0, 1.0))))

def compute_angles(kp):
    """12-dim angle feature vector (6 angles + 3 diffs + 3 means)."""
    xy, vis = kp[:, :2], kp[:, 3]
    def safe(a, b, c):
        return (_angle_3pts_2d(xy[a], xy[b], xy[c])
                if min(vis[a], vis[b], vis[c]) >= VIS_THRESH else 0.0)
    l_el = safe(LS, LE, LW);  r_el = safe(RS, RE, RW)
    l_kn = safe(LH, LK, LA);  r_kn = safe(RH, RK, RA)
    l_hp = safe(LS, LH, LK);  r_hp = safe(RS, RH, RK)
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el - r_el, l_kn - r_kn, l_hp - r_hp,
                     (l_el + r_el)/2, (l_kn + r_kn)/2, (l_hp + r_hp)/2],
                    dtype=np.float32)

def clip_to_features(clip_norm):
    """Normalized (T, 33, 4) clip -> (T, 210) feature matrix."""
    T  = len(clip_norm)
    xy = clip_norm[:, :, :2].reshape(T, -1)    # 66
    z  = clip_norm[:, :, 2].reshape(T, -1)     # 33
    vis= clip_norm[:, :, 3].reshape(T, -1)     # 33
    vxy = np.zeros_like(xy)
    vxy[1:] = xy[1:] - xy[:-1]                 # 66
    ang = np.stack([compute_angles(clip_norm[t]) for t in range(T)])  # 12
    feat = np.concatenate([xy, vxy, z, vis, ang], axis=1).astype(np.float32)
    assert feat.shape == (T, FEAT_DIM)
    return feat

print("Normalization + feature helpers ready")


In [ ]:
def _rotate_chain_2d(clip, vertex_idx, downstream_ids, angle_deg):
    """
    Rotate `downstream_ids` landmarks around `vertex_idx` by `angle_deg` (in XY)
    consistently across all frames. z and visibility untouched.
    Mutates and returns `clip` — shape (T, 33, 4).
    """
    theta = np.deg2rad(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    # Per-frame vertex position — rotation pivot drifts with the person
    for t in range(len(clip)):
        vx, vy = clip[t, vertex_idx, 0], clip[t, vertex_idx, 1]
        for lm_id in downstream_ids:
            x, y = clip[t, lm_id, 0], clip[t, lm_id, 1]
            dx, dy = x - vx, y - vy
            clip[t, lm_id, 0] = vx + cos_t * dx - sin_t * dy
            clip[t, lm_id, 1] = vy + sin_t * dx + cos_t * dy
    return clip

def corrupt_clip(clip_norm, rng):
    """
    Produce a synthetically-incorrect twin of a clean normalized clip.
    Returns (corrupted_clip, feedback_vec) where feedback_vec is a (6,) array
    of applied corruption magnitude in degrees (zero for uncorrupted joints).
    """
    out = clip_norm.copy()

    # Pick how many joints to corrupt, then which ones (without replacement)
    n_corrupt = rng.integers(N_CORRUPT_JOINTS_MIN, N_CORRUPT_JOINTS_MAX + 1)
    joints = rng.choice(N_JOINTS, size=n_corrupt, replace=False)

    feedback = np.zeros(N_JOINTS, dtype=np.float32)
    for j in joints:
        name  = JOINT_NAMES[j]
        # Signed angle so rotation direction isn't learnable from sign
        mag   = rng.uniform(CORRUPT_ANGLE_MIN_DEG, CORRUPT_ANGLE_MAX_DEG)
        sign  = rng.choice([-1.0, 1.0])
        angle = sign * mag
        _rotate_chain_2d(out, JOINT_VERTEX[name], CHAIN_DOWNSTREAM[name], angle)
        feedback[j] = mag  # store unsigned magnitude (matches Stage 3/4 semantics)

    # Global jitter on all landmarks after rotation, XY only
    jitter = rng.normal(0.0, JITTER_SIGMA, size=out[..., :2].shape).astype(np.float32)
    out[..., :2] += jitter
    return out, feedback

def clean_feedback():
    """Feedback ground truth for clean samples — all zeros."""
    return np.zeros(N_JOINTS, dtype=np.float32)

print("Corruption helpers ready")


In [ ]:
all_X, all_q, all_p, all_fb, all_meta = [], [], [], [], []
skipped_bad_pose = skipped_no_windows = 0

for _, row in meta.iterrows():
    pose_name = row["pose"]
    if pose_name not in POSE_TO_ID:
        skipped_bad_pose += 1
        continue
    pose_id = POSE_TO_ID[pose_name]

    clip_raw = np.load(row["clip_path"])           # (T, 33, 4)
    clip_norm = normalize_clip(clip_raw)

    # Clean features + corrupted features (clip-level corruption = consistent
    # across all frames of this clip)
    feat_clean = clip_to_features(clip_norm)
    clip_corrupt, fb_vec = corrupt_clip(clip_norm, RNG)
    feat_corrupt = clip_to_features(clip_corrupt)

    # Sanity: both features should have identical length — they're same T
    assert feat_clean.shape == feat_corrupt.shape

    # Slide windows over BOTH feature streams at the same offsets
    T = feat_clean.shape[0]
    starts = list(range(0, T - WINDOW_SIZE + 1, STRIDE))
    if not starts:
        skipped_no_windows += 1
        continue

    # Detection mask lives in columns 132:165 (visibility block of 210-dim vec)
    detected = (feat_clean[:, 132:165].max(axis=1) > 0)   # (T,)

    for start in starts:
        end = start + WINDOW_SIZE
        if detected[start:end].mean() < MIN_VIS_FRAC:
            continue

        # Clean twin
        all_X.append(feat_clean[start:end])
        all_q.append(LABEL_CORRECT)
        all_p.append(pose_id)
        all_fb.append(clean_feedback())
        all_meta.append({
            "clip_index": int(row["clip_index"]),
            "pose": pose_name,
            "quality_label": "correct",
            "window_start": start,
            "source_file": row["source_file"],
            "dataset": "3dyoga90",
        })

        # Corrupted twin — same parent clip, same window start, opposite quality
        all_X.append(feat_corrupt[start:end])
        all_q.append(LABEL_INCORRECT)
        all_p.append(pose_id)
        all_fb.append(fb_vec)
        all_meta.append({
            "clip_index": int(row["clip_index"]),
            "pose": pose_name,
            "quality_label": "incorrect",
            "window_start": start,
            "source_file": row["source_file"],
            "dataset": "3dyoga90",
        })

print(f"Clips skipped (bad pose)  : {skipped_bad_pose}")
print(f"Clips skipped (no window) : {skipped_no_windows}")
print(f"Total windows: {len(all_X)}  (correct={sum(q==0 for q in all_q)}, "
      f"incorrect={sum(q==1 for q in all_q)})")


In [ ]:
from sklearn.model_selection import train_test_split

meta_df = pd.DataFrame(all_meta)
clip_ids = meta_df["clip_index"].unique()
# Stratify clip-level split by the clip's pose so each split gets all 7 classes
clip_pose = {cid: meta_df[meta_df["clip_index"] == cid]["pose"].iloc[0]
             for cid in clip_ids}
clip_pose_labels = [POSE_TO_ID[clip_pose[c]] for c in clip_ids]

train_cids, temp_cids, _, temp_lbls = train_test_split(
    clip_ids, clip_pose_labels, test_size=0.30, random_state=42,
    stratify=clip_pose_labels)
val_cids, test_cids = train_test_split(
    temp_cids, test_size=0.50, random_state=42, stratify=temp_lbls)

train_set, val_set, test_set = set(train_cids), set(val_cids), set(test_cids)

def split_of(cid):
    if cid in train_set: return "train"
    if cid in val_set:   return "val"
    return "test"

# Assign every window to a split
X_train, X_val, X_test = [], [], []
q_train, q_val, q_test = [], [], []
p_train, p_val, p_test = [], [], []
fb_train, fb_val, fb_test = [], [], []
mt_train, mt_val, mt_test = [], [], []

for X, q, p, fb, m in zip(all_X, all_q, all_p, all_fb, all_meta):
    bucket = split_of(m["clip_index"])
    if bucket == "train":
        X_train.append(X); q_train.append(q); p_train.append(p)
        fb_train.append(fb); mt_train.append(m)
    elif bucket == "val":
        X_val.append(X); q_val.append(q); p_val.append(p)
        fb_val.append(fb); mt_val.append(m)
    else:
        X_test.append(X); q_test.append(q); p_test.append(p)
        fb_test.append(fb); mt_test.append(m)

X_train = np.stack(X_train).astype(np.float32)
X_val   = np.stack(X_val).astype(np.float32)
X_test  = np.stack(X_test).astype(np.float32)
y_quality_train = np.array(q_train, dtype=np.int64)
y_quality_val   = np.array(q_val,   dtype=np.int64)
y_quality_test  = np.array(q_test,  dtype=np.int64)
y_pose_train = np.array(p_train, dtype=np.int64)
y_pose_val   = np.array(p_val,   dtype=np.int64)
y_pose_test  = np.array(p_test,  dtype=np.int64)
y_fb_train = np.stack(fb_train).astype(np.float32)
y_fb_val   = np.stack(fb_val).astype(np.float32)
y_fb_test  = np.stack(fb_test).astype(np.float32)

# Feature stats — train only (clean + corrupted together, they share a distribution
# after normalization; normalizing on clean-only would bias the jitter scale)
N, T, D   = X_train.shape
feat_mean = X_train.reshape(-1, D).mean(axis=0).astype(np.float32)
feat_std  = X_train.reshape(-1, D).std(axis=0).astype(np.float32) + 1e-6
np.savez(f"{OUT}/feature_stats.npz", mean=feat_mean, std=feat_std)

# Save everything
np.save(f"{OUT}/X_train.npy", X_train); np.save(f"{OUT}/X_val.npy", X_val); np.save(f"{OUT}/X_test.npy", X_test)
np.save(f"{OUT}/y_quality_train.npy", y_quality_train)
np.save(f"{OUT}/y_quality_val.npy",   y_quality_val)
np.save(f"{OUT}/y_quality_test.npy",  y_quality_test)
np.save(f"{OUT}/y_pose_train.npy", y_pose_train)
np.save(f"{OUT}/y_pose_val.npy",   y_pose_val)
np.save(f"{OUT}/y_pose_test.npy",  y_pose_test)
np.save(f"{OUT}/y_feedback_train.npy", y_fb_train)
np.save(f"{OUT}/y_feedback_val.npy",   y_fb_val)
np.save(f"{OUT}/y_feedback_test.npy",  y_fb_test)
pd.DataFrame(mt_train).to_csv(f"{OUT}/meta_train.csv", index=False)
pd.DataFrame(mt_val).to_csv(  f"{OUT}/meta_val.csv",   index=False)
pd.DataFrame(mt_test).to_csv( f"{OUT}/meta_test.csv",  index=False)

# Summary
print("=" * 55)
print("Stage 2 complete")
print(f"  Train: {X_train.shape}  | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"  NaN in X_train: {np.isnan(X_train).sum()}")
print(f"  Train quality balance: correct={(y_quality_train==0).sum()} "
      f"incorrect={(y_quality_train==1).sum()}")
print("  Train pose distribution:")
for pid, pname in enumerate(POSE_NAMES):
    print(f"    [{pid}] {pname:22s} {(y_pose_train==pid).sum()}")
print(f"  Feedback target range (train): "
      f"min={y_fb_train.min():.2f}, max={y_fb_train.max():.2f}, "
      f"mean_nonzero={y_fb_train[y_fb_train>0].mean():.2f} deg")
print(f"  Feature stats saved: {OUT}/feature_stats.npz")


In [ ]:
import numpy as np
import pandas as pd
import os, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

OUT = "path/to/working/windows"

X_train = np.load(f"{OUT}/X_train.npy")
X_val   = np.load(f"{OUT}/X_val.npy")
X_test  = np.load(f"{OUT}/X_test.npy")

y_q_train = np.load(f"{OUT}/y_quality_train.npy")
y_q_val   = np.load(f"{OUT}/y_quality_val.npy")
y_q_test  = np.load(f"{OUT}/y_quality_test.npy")
y_p_train = np.load(f"{OUT}/y_pose_train.npy")
y_p_val   = np.load(f"{OUT}/y_pose_val.npy")
y_p_test  = np.load(f"{OUT}/y_pose_test.npy")
y_fb_train = np.load(f"{OUT}/y_feedback_train.npy")
y_fb_val   = np.load(f"{OUT}/y_feedback_val.npy")
y_fb_test  = np.load(f"{OUT}/y_feedback_test.npy")

stats     = np.load(f"{OUT}/feature_stats.npz")
FEAT_MEAN = stats["mean"]
FEAT_STD  = stats["std"]

POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]
N_POSES    = 7
N_JOINTS   = 6
JOINT_NAMES = ["left_elbow", "right_elbow", "left_knee",
               "right_knee", "left_hip",    "right_hip"]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Quality balance train: correct={(y_q_train==0).sum()} "
      f"incorrect={(y_q_train==1).sum()}")


In [ ]:
class YogaWindowDataset(Dataset):
    def __init__(self, X, y_q, y_p, y_fb, mean, std):
        X_std = (X - mean[None, None, :]) / std[None, None, :]
        self.X    = torch.from_numpy(X_std).float()
        self.y_q  = torch.from_numpy(y_q).long()
        self.y_p  = torch.from_numpy(y_p).long()
        self.y_fb = torch.from_numpy(y_fb).float()

    def __len__(self):  return len(self.y_q)
    def __getitem__(self, i):
        return self.X[i], self.y_q[i], self.y_p[i], self.y_fb[i]

train_ds = YogaWindowDataset(X_train, y_q_train, y_p_train, y_fb_train,
                             FEAT_MEAN, FEAT_STD)
val_ds   = YogaWindowDataset(X_val,   y_q_val,   y_p_val,   y_fb_val,
                             FEAT_MEAN, FEAT_STD)
test_ds  = YogaWindowDataset(X_test,  y_q_test,  y_p_test,  y_fb_test,
                             FEAT_MEAN, FEAT_STD)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
class TCNBlock(nn.Module):
    """Causal TCN — left-pad only so output length == input length."""
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim=210, hidden=128, layers=6,
                 n_joints=6, n_poses=7):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)

        # Soft conditioning: initialized to zeros so it has no effect at start
        self.cond_matrix = nn.Parameter(torch.zeros(n_poses, n_joints))

    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))                  # (B, C, T)
        g = self.pool(h).squeeze(-1)                     # (B, hidden)
        quality_logits = self.quality_head(g)            # (B, 2)
        pose_logits    = self.pose_head(g)               # (B, 7)
        feedback_raw   = self.feedback_head(g)           # (B, 6)
        pose_probs     = F.softmax(pose_logits, dim=1)
        cond           = pose_probs @ self.cond_matrix
        feedback       = F.relu(feedback_raw + cond)     # non-negative degrees
        return quality_logits, pose_logits, feedback

FEAT_DIM = X_train.shape[2]
model = YogaTCN(feat_dim=FEAT_DIM, hidden=128, layers=6,
                n_joints=N_JOINTS, n_poses=N_POSES).to(device)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total:,}")


In [ ]:
QUALITY_W = 1.0
POSE_W    = 0.5
FB_W      = 0.4

criterion_q = nn.CrossEntropyLoss()

# Optional per-pose inverse-frequency weights for the pose head.
# Uncomment the `weight=` argument if mountain (or another class) lags.
pose_counts  = np.bincount(y_p_train, minlength=N_POSES).astype(np.float32)
pose_weights = pose_counts.mean() / pose_counts                     # inverse freq
pose_weights_t = torch.from_numpy(pose_weights).to(device)
print("Per-pose inverse-frequency weights (ref only — disabled by default):")
for pid, pname in enumerate(POSE_NAMES):
    print(f"  [{pid}] {pname:22s} count={int(pose_counts[pid]):5d} "
          f"weight={pose_weights[pid]:.3f}")

criterion_p = nn.CrossEntropyLoss()  # add `weight=pose_weights_t` to weight
# Feedback loss is smooth-L1 applied functionally in the training loop

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=5, factor=0.5
)
print("Loss, optimizer, scheduler ready")


In [ ]:
CHECKPOINT = "path/to/working/yoga_tcn_pretrain_best.pt"
MAX_EPOCHS = 80
PATIENCE   = 10

def eval_quality_acc(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb_q, _, _ in loader:
            Xb, yb_q = Xb.to(device), yb_q.to(device)
            logits_q, _, _ = model(Xb)
            correct += (logits_q.argmax(1) == yb_q).sum().item()
            total   += yb_q.size(0)
    return correct / total

best_val_acc = 0.0
bad_epochs   = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_seen = 0

    for Xb, yb_q, yb_p, yb_fb in train_loader:
        Xb    = Xb.to(device, non_blocking=True)
        yb_q  = yb_q.to(device, non_blocking=True)
        yb_p  = yb_p.to(device, non_blocking=True)
        yb_fb = yb_fb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits_q, logits_p, feedback = model(Xb)

        loss_q  = criterion_q(logits_q, yb_q)
        loss_p  = criterion_p(logits_p, yb_p)
        loss_fb = F.smooth_l1_loss(feedback, yb_fb)
        loss    = QUALITY_W * loss_q + POSE_W * loss_p + FB_W * loss_fb

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * yb_q.size(0)
        n_seen       += yb_q.size(0)

    train_loss = running_loss / n_seen
    val_acc    = eval_quality_acc(val_loader)
    scheduler.step(val_acc)

    print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_quality_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        bad_epochs   = 0
        torch.save(model.state_dict(), CHECKPOINT)
        print("  ✅ saved best checkpoint")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("  🛑 early stopping")
            break

print(f"\nBest val quality accuracy: {best_val_acc:.4f}")


In [ ]:
# 20 minutes training on kaggle CPU over 20 epochs


In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()

all_q_preds, all_q_true   = [], []
all_p_preds, all_p_true   = [], []
all_fb_preds, all_fb_true = [], []

with torch.no_grad():
    for Xb, yb_q, yb_p, yb_fb in test_loader:
        Xb = Xb.to(device)
        logits_q, logits_p, feedback = model(Xb)
        all_q_preds.append(logits_q.argmax(1).cpu().numpy())
        all_q_true.append(yb_q.numpy())
        all_p_preds.append(logits_p.argmax(1).cpu().numpy())
        all_p_true.append(yb_p.numpy())
        all_fb_preds.append(feedback.cpu().numpy())
        all_fb_true.append(yb_fb.numpy())

all_q_preds = np.concatenate(all_q_preds);  all_q_true = np.concatenate(all_q_true)
all_p_preds = np.concatenate(all_p_preds);  all_p_true = np.concatenate(all_p_true)
all_fb_preds = np.concatenate(all_fb_preds); all_fb_true = np.concatenate(all_fb_true)

print("=" * 55)
print("QUALITY HEAD")
print(f"Test accuracy: {(all_q_preds == all_q_true).mean():.4f}")
print(confusion_matrix(all_q_true, all_q_preds))
print(classification_report(all_q_true, all_q_preds,
                            target_names=["correct", "incorrect"]))

print("=" * 55)
print("POSE HEAD (7 classes)")
print(f"Test accuracy: {(all_p_preds == all_p_true).mean():.4f}")
print(classification_report(all_p_true, all_p_preds,
                            target_names=POSE_NAMES, digits=3))

print("=" * 55)
print("FEEDBACK HEAD — per-joint MAE (degrees)")
for j, name in enumerate(JOINT_NAMES):
    mae = np.abs(all_fb_preds[:, j] - all_fb_true[:, j]).mean()
    print(f"  {name:20s} MAE={mae:5.2f} deg")

# Pose-angle reference means — used by Stage 4's inference for ideal-skeleton
# projection. Computed on clean (correct) training windows only.
ANGLE_START = 198
ANGLE_COLS  = 6
clean_mask  = (y_q_train == 0)
pose_angle_means = {}
pose_landmark_means = {}
for pid in range(N_POSES):
    m = (y_p_train == pid) & clean_mask
    if m.sum() > 0:
        # X_train is already standardized in the dataset but the arrays on disk
        # are NOT — we reload normalized-but-not-standardized features here.
        # Feature columns 0..66 are hip-centered shoulder-scaled xy.
        angles = X_train[m, :, ANGLE_START:ANGLE_START+ANGLE_COLS].mean(axis=(0,1))
        pose_angle_means[pid] = angles.astype(np.float32)
        xy     = X_train[m, :, 0:66].mean(axis=(0,1)).reshape(33, 2)
        pose_landmark_means[pid] = xy.astype(np.float32)
    else:
        pose_angle_means[pid]    = np.zeros(ANGLE_COLS, dtype=np.float32)
        pose_landmark_means[pid] = np.zeros((33, 2),    dtype=np.float32)

# Save everything Stages 4–7 need
torch.save({
    "model":               model.state_dict(),
    "feat_dim":            int(FEAT_DIM),
    "hidden":              128,
    "layers":              6,
    "n_joints":            N_JOINTS,
    "n_poses":             N_POSES,
    "joint_names":         JOINT_NAMES,
    "pose_names":          POSE_NAMES,
    "pose_to_id":          {p: i for i, p in enumerate(POSE_NAMES)},
    "feat_mean":           FEAT_MEAN.tolist(),
    "feat_std":            FEAT_STD.tolist(),
    "pose_angle_means":    {k: v.tolist() for k, v in pose_angle_means.items()},
    "pose_landmark_means": {k: v.tolist() for k, v in pose_landmark_means.items()},
    "max_feedback_deg":    45.0,
    "angle_start":         ANGLE_START,
    "angle_cols":          ANGLE_COLS,
    "window_size":         int(X_train.shape[1]),
}, "path/to/working/yoga_tcn_pretrain.pt")

print("\n✅ Pretrain checkpoint saved: path/to/working/yoga_tcn_pretrain.pt")
print("   Ready for Stage 4 (YogNet extraction) and Stage 5 (cross-dataset test)")


In [ ]:
# MediaPipe setup for Stage 4 video extraction (only needed for RGB video input).
import os, urllib.request

os.makedirs("path/to/working/models", exist_ok=True)

MODEL_PATH = "path/to/working/models/pose_landmarker_full.task"
URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(URL, MODEL_PATH)
print(f"Model: {MODEL_PATH}  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")


In [ ]:
# Stage 4 - YogNet video extraction: run MediaPipe on RGB videos, write (T,33,4) clips.


In [ ]:
# ===================== CONFIG — update YOGNET_ROOT ======================
YOGNET_ROOT   = "path/to/yognet7poses"
YOGNET_OUT    = "path/to/working/extracted_yognet"
FRAME_STRIDE  = 2            # agreed half-rate extraction
MIN_FRAMES    = 33           # same floor as Stage 2
VIDEO_EXTS    = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
# ========================================================================

# YogNet folder name -> 7-target canonical name
YOGNET_POSE_MAP = {
    "Adhomukhasvanasana": "downward_facing_dog",
    "Bhujangasana":       "cobra",
    "Shavasana":           "corpse",
    "Tadasana":            "mountain",
    "Trikonasana":         "triangle",
    "Virbhadrasana1":      "warrior",
    "Vrikshasana":         "tree",
}

os.makedirs(f"{YOGNET_OUT}/clips", exist_ok=True)
print(f"YogNet root : {YOGNET_ROOT}")
print(f"Output      : {YOGNET_OUT}")
print(f"Folder->pose map ({len(YOGNET_POSE_MAP)} entries):")
for folder, pose in YOGNET_POSE_MAP.items():
    print(f"  {folder:25s} -> {pose}")


In [ ]:
if not os.path.isdir(YOGNET_ROOT):
    raise FileNotFoundError(f"YOGNET_ROOT does not exist: {YOGNET_ROOT}")

found_folders = sorted(
    d for d in os.listdir(YOGNET_ROOT)
    if os.path.isdir(os.path.join(YOGNET_ROOT, d))
)
print(f"Found {len(found_folders)} folders under YOGNET_ROOT:")
for d in found_folders:
    vids = [f for f in os.listdir(os.path.join(YOGNET_ROOT, d))
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS]
    mapped = YOGNET_POSE_MAP.get(d, "— NOT IN MAP —")
    print(f"  {d:25s} [{len(vids):3d} videos]  -> {mapped}")

# Complain about unmapped folders or missing expected folders
unexpected = [d for d in found_folders if d not in YOGNET_POSE_MAP]
missing    = [d for d in YOGNET_POSE_MAP if d not in found_folders]
if unexpected:
    print(f"\n⚠️  Unexpected folders (will be skipped): {unexpected}")
if missing:
    print(f"\n⚠️  Missing expected folders: {missing}")


In [ ]:
metadata_rows   = []
clip_counter    = 0
skipped_unread  = 0
skipped_short   = 0
skipped_unmap   = 0

for folder in sorted(os.listdir(YOGNET_ROOT)):
    folder_path = os.path.join(YOGNET_ROOT, folder)
    if not os.path.isdir(folder_path):
        continue

    pose_name = YOGNET_POSE_MAP.get(folder)
    if pose_name is None:
        skipped_unmap += 1
        continue

    video_files = sorted(
        f for f in os.listdir(folder_path)
        if os.path.splitext(f)[1].lower() in VIDEO_EXTS
    )
    print(f"\n[{folder}] -> {pose_name} | {len(video_files)} videos")

    for vf in video_files:
        vpath = os.path.join(folder_path, vf)
        clip, stats = process_video(vpath, landmarker, stride=FRAME_STRIDE)

        if clip is None:
            skipped_unread += 1
            print(f"  [skip] {vf} — unreadable")
            continue
        if stats.get("frames_kept", 0) < MIN_FRAMES:
            skipped_short += 1
            print(f"  [skip] {vf} — only {stats.get('frames_kept', 0)} frames")
            continue

        out_name = f"{pose_name}__correct__{clip_counter:04d}.npy"
        out_path = os.path.join(YOGNET_OUT, "clips", out_name)
        np.save(out_path, clip)

        metadata_rows.append({
            "clip_index":      clip_counter,
            "pose":            pose_name,
            "original_label":  folder,
            "quality_label":   "correct",         # YogNet is all demonstrations
            "frames_kept":     stats["frames_kept"],
            "frames_detected": stats["frames_detected"],
            "detection_rate":  stats["detection_rate"],
            "clip_path":       out_path,
            "source_file":     vpath,
            "split":           "",                # no official split; built in Stage 6
            "dataset":         "yognet",
        })
        clip_counter += 1
        print(f"  {vf}: {stats['frames_kept']} frames, "
              f"det={stats['detection_rate']:.2%}")

print(f"\n\nStage 4 extraction done:")
print(f"  clips saved         : {clip_counter}")
print(f"  skipped (short)     : {skipped_short}  (<{MIN_FRAMES} frames)")
print(f"  skipped (unreadable): {skipped_unread}")
print(f"  skipped (unmapped)  : {skipped_unmap}")


In [ ]:
fieldnames = ["clip_index", "pose", "original_label", "quality_label",
              "frames_kept", "frames_detected", "detection_rate",
              "clip_path", "source_file", "split", "dataset"]

meta_path = f"{YOGNET_OUT}/metadata.csv"
with open(meta_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(metadata_rows)

print(f"Metadata saved: {meta_path}")


In [ ]:
meta = pd.read_csv(meta_path)

print("=" * 55)
print(f"Total clips: {len(meta)}\n")

print("Clips per target pose:")
print(meta.groupby("pose")["clip_index"].count().to_string())

print("\nDetection rate by pose:")
det = (
    meta.groupby("pose")["detection_rate"]
        .agg(["mean", "min", "count"])
        .rename(columns={"mean": "mean_det", "min": "min_det", "count": "n_clips"})
        .sort_values("mean_det")
)
det["mean_det"] = det["mean_det"].map("{:.2%}".format)
det["min_det"]  = det["min_det"].map("{:.2%}".format)
print(det.to_string())

print("\nFrames per clip:")
print(meta["frames_kept"].describe().round(1).to_string())


In [ ]:
WINDOW_SIZE = 32
STRIDE      = 32

def _n_windows(T):
    if T < WINDOW_SIZE:
        return 0
    return (T - WINDOW_SIZE) // STRIDE + 1

meta["est_windows"] = meta["frames_kept"].apply(_n_windows)
print("Estimated windows per pose (before MIN_VIS_FRAC filter):")
print(meta.groupby("pose")["est_windows"].agg(["sum", "mean"])
         .rename(columns={"sum": "total_windows",
                          "mean": "mean_windows_per_clip"})
         .round(1).to_string())

total = meta["est_windows"].sum()
print(f"\nExpected YogNet windows total : {total}")
print(f"After 1:1 Stage 6 augmentation: ~{total * 2}")


In [ ]:
# Stage 5 — Cross-dataset test: pretrained model on YogNet (no training)
import os, torch, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report

# ---- Load pretrained checkpoint ----
CKPT_PATH = "path/to/working/yoga_tcn_pretrain.pt"
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

FEAT_DIM     = ckpt["feat_dim"]
HIDDEN       = ckpt["hidden"]
LAYERS       = ckpt["layers"]
N_JOINTS     = ckpt["n_joints"]
N_POSES      = ckpt["n_poses"]
POSE_NAMES   = ckpt["pose_names"]
POSE_TO_ID   = ckpt["pose_to_id"]
JOINT_NAMES  = ckpt["joint_names"]
WINDOW_SIZE  = ckpt["window_size"]
FEAT_MEAN    = np.array(ckpt["feat_mean"], dtype=np.float32)
FEAT_STD     = np.array(ckpt["feat_std"],  dtype=np.float32)

print(f"Pretrain checkpoint loaded:")
print(f"  Poses ({N_POSES}): {POSE_NAMES}")
print(f"  Window size: {WINDOW_SIZE} | Feat dim: {FEAT_DIM}")

# ---- Load YogNet metadata ----
YOGNET_META = pd.read_csv("path/to/working/extracted_yognet/metadata.csv")
print(f"\nYogNet clips: {len(YOGNET_META)}")
print(YOGNET_META.groupby("pose")["clip_index"].count().to_string())


In [ ]:
# Same config as Stage 2's windowing
STRIDE         = 32
MIN_VIS_FRAC   = 0.6

# Landmark indices + normalization (mirrors Stage 2 Cell 3 helpers)
LS, RS = 11, 12
LH, RH = 23, 24
VIS_THRESH = 0.3
LE, RE = 13, 14; LW, RW = 15, 16
LK, RK = 25, 26; LA, RA = 27, 28

def normalize_clip(clip):
    out = clip.copy()
    for t in range(len(out)):
        hip = (out[t, LH, :3] + out[t, RH, :3]) / 2.0
        out[t, :, :3] -= hip
        scale = np.linalg.norm(out[t, LS, :3] - out[t, RS, :3]) + 1e-6
        out[t, :, :3] /= scale
    return out

def _angle_3pts_2d(a, b, c):
    ba = a - b; bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    return float(np.degrees(np.arccos(np.clip(np.dot(ba, bc)/denom, -1.0, 1.0))))

def compute_angles(kp):
    xy, vis = kp[:, :2], kp[:, 3]
    def safe(a, b, c):
        return (_angle_3pts_2d(xy[a], xy[b], xy[c])
                if min(vis[a], vis[b], vis[c]) >= VIS_THRESH else 0.0)
    l_el = safe(LS, LE, LW);  r_el = safe(RS, RE, RW)
    l_kn = safe(LH, LK, LA);  r_kn = safe(RH, RK, RA)
    l_hp = safe(LS, LH, LK);  r_hp = safe(RS, RH, RK)
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el-r_el, l_kn-r_kn, l_hp-r_hp,
                     (l_el+r_el)/2, (l_kn+r_kn)/2, (l_hp+r_hp)/2],
                    dtype=np.float32)

def clip_to_features(clip_norm):
    T = len(clip_norm)
    xy = clip_norm[:, :, :2].reshape(T, -1)
    z  = clip_norm[:, :, 2].reshape(T, -1)
    vis = clip_norm[:, :, 3].reshape(T, -1)
    vxy = np.zeros_like(xy); vxy[1:] = xy[1:] - xy[:-1]
    ang = np.stack([compute_angles(clip_norm[t]) for t in range(T)])
    return np.concatenate([xy, vxy, z, vis, ang], axis=1).astype(np.float32)

# Build windows
all_X, all_pose, all_clip_id, all_src_file = [], [], [], []
per_pose_kept = {p: 0 for p in POSE_NAMES}
per_pose_filtered_out = {p: 0 for p in POSE_NAMES}

for _, row in YOGNET_META.iterrows():
    pose_name = row["pose"]
    if pose_name not in POSE_TO_ID:
        continue
    pose_id = POSE_TO_ID[pose_name]

    clip_raw = np.load(row["clip_path"])
    clip_norm = normalize_clip(clip_raw)
    feat = clip_to_features(clip_norm)
    T = feat.shape[0]

    detected = (feat[:, 132:165].max(axis=1) > 0)

    for start in range(0, T - WINDOW_SIZE + 1, STRIDE):
        end = start + WINDOW_SIZE
        if detected[start:end].mean() < MIN_VIS_FRAC:
            per_pose_filtered_out[pose_name] += 1
            continue
        all_X.append(feat[start:end])
        all_pose.append(pose_id)
        all_clip_id.append(int(row["clip_index"]))
        all_src_file.append(row["source_file"])
        per_pose_kept[pose_name] += 1

X_test  = np.stack(all_X).astype(np.float32)
y_pose_test = np.array(all_pose, dtype=np.int64)

print(f"YogNet windows kept after MIN_VIS_FRAC={MIN_VIS_FRAC}:")
for p in POSE_NAMES:
    kept, dropped = per_pose_kept[p], per_pose_filtered_out[p]
    total = kept + dropped
    pct = kept/total*100 if total > 0 else 0
    print(f"  {p:22s} kept={kept:4d}  dropped={dropped:4d}  ({pct:.0f}% survive)")
print(f"\n  TOTAL: {len(X_test)} windows for evaluation")
print(f"  X_test shape: {X_test.shape}")


In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim, hidden, layers, n_joints, n_poses):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)
        self.cond_matrix   = nn.Parameter(torch.zeros(n_poses, n_joints))
    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))
        g = self.pool(h).squeeze(-1)
        qs = self.quality_head(g)
        ps = self.pose_head(g)
        fr = self.feedback_head(g)
        pp = F.softmax(ps, dim=1)
        fb = F.relu(fr + pp @ self.cond_matrix)
        return qs, ps, fb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YogaTCN(FEAT_DIM, HIDDEN, LAYERS, N_JOINTS, N_POSES).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Model ready on {device}")


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, mean, std):
        X_std = (X - mean[None, None, :]) / std[None, None, :]
        self.X = torch.from_numpy(X_std).float()
    def __len__(self):  return len(self.X)
    def __getitem__(self, i):  return self.X[i]

ds = WindowDataset(X_test, FEAT_MEAN, FEAT_STD)
loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=2)

all_q_preds, all_q_probs = [], []
all_p_preds, all_p_probs = [], []
all_fb_preds = []

with torch.no_grad():
    for Xb in loader:
        Xb = Xb.to(device)
        logits_q, logits_p, feedback = model(Xb)
        q_probs = F.softmax(logits_q, dim=1)
        p_probs = F.softmax(logits_p, dim=1)
        all_q_preds.append(logits_q.argmax(1).cpu().numpy())
        all_q_probs.append(q_probs.cpu().numpy())
        all_p_preds.append(logits_p.argmax(1).cpu().numpy())
        all_p_probs.append(p_probs.cpu().numpy())
        all_fb_preds.append(feedback.cpu().numpy())

all_q_preds  = np.concatenate(all_q_preds)
all_q_probs  = np.concatenate(all_q_probs)
all_p_preds  = np.concatenate(all_p_preds)
all_p_probs  = np.concatenate(all_p_probs)
all_fb_preds = np.concatenate(all_fb_preds)

print(f"Forward pass complete on {len(all_q_preds)} windows")


In [ ]:
print("=" * 60)
print("POSE HEAD — transfer from 3DYoga90 to YogNet")
print(f"Overall accuracy: {(all_p_preds == y_pose_test).mean():.4f}")
print(confusion_matrix(y_pose_test, all_p_preds))
print(classification_report(y_pose_test, all_p_preds,
                            target_names=POSE_NAMES, digits=3))

print("=" * 60)
print("QUALITY HEAD — (YogNet is all clean; expected to predict 'correct')")
print(f"Fraction predicted 'correct' (label 0): "
      f"{(all_q_preds == 0).mean():.4f}")
print(f"Fraction predicted 'incorrect' (label 1): "
      f"{(all_q_preds == 1).mean():.4f}")
print(f"Mean P(correct): {all_q_probs[:, 0].mean():.3f}  "
      f"  median: {np.median(all_q_probs[:, 0]):.3f}")
print(f"Mean P(incorrect): {all_q_probs[:, 1].mean():.3f}")
print("\nP(correct) distribution by predicted pose:")
for pid, pname in enumerate(POSE_NAMES):
    m = (all_p_preds == pid)
    if m.sum() == 0:
        continue
    mean_pc = all_q_probs[m, 0].mean()
    n_correct_pred = (all_q_preds[m] == 0).sum()
    print(f"  {pname:22s} n={m.sum():4d}  "
          f"P(correct)_mean={mean_pc:.3f}  "
          f"pred_correct={n_correct_pred}/{m.sum()}")

print("=" * 60)
print("FEEDBACK HEAD — MAE against zero targets (all YogNet is clean)")
for j, name in enumerate(JOINT_NAMES):
    mae = np.abs(all_fb_preds[:, j]).mean()
    print(f"  {name:20s} MAE={mae:5.2f} deg  (ideal=0.0)")

# Per-pose feedback magnitude — which poses does pretrain think have errors?
print("\nMean feedback magnitude per pose (across all 6 joints):")
for pid, pname in enumerate(POSE_NAMES):
    m = (y_pose_test == pid)
    if m.sum() == 0:
        continue
    mean_fb = np.abs(all_fb_preds[m]).mean()
    print(f"  {pname:22s} n={m.sum():4d}  mean_|fb|={mean_fb:5.2f} deg")


In [ ]:
# Stage 6 — Fine-tune pretrained model on YogNet with synthetic-incorrect twins
import os, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Windowing — match Stage 2/5 so windows are comparable
WINDOW_SIZE  = 32
STRIDE       = 32
MIN_VIS_FRAC = 0.6
MIN_FRAMES   = 33

# Augmentation — same scheme as Stage 2
N_CORRUPT_JOINTS_MIN  = 1
N_CORRUPT_JOINTS_MAX  = 2
CORRUPT_ANGLE_MIN_DEG = 0.0
CORRUPT_ANGLE_MAX_DEG = 45.0
JITTER_SIGMA          = 0.01

# Fine-tune training
FT_LR         = 3e-4
QUALITY_W_FT  = 1.0
POSE_W_FT     = 1.0
FB_W_FT       = 0.5
MAX_EPOCHS    = 40
PATIENCE      = 8
BATCH_SIZE    = 64

# Unfreeze policy: last N TCN blocks + all heads + cond_matrix
N_UNFROZEN_TCN_BLOCKS = 2

RNG = np.random.default_rng(123)  # different seed from Stage 2

CKPT_PRETRAIN = "path/to/working/yoga_tcn_pretrain.pt"
CKPT_BEST_FT  = "path/to/working/yoga_tcn_finetune_best.pt"
CKPT_FULL_FT  = "path/to/working/yoga_tcn_finetuned.pt"
print("Stage 6 config set")


In [ ]:
ckpt = torch.load(CKPT_PRETRAIN, map_location="cpu", weights_only=False)

FEAT_DIM     = ckpt["feat_dim"]
HIDDEN       = ckpt["hidden"]
LAYERS       = ckpt["layers"]
N_JOINTS     = ckpt["n_joints"]
N_POSES      = ckpt["n_poses"]
POSE_NAMES   = ckpt["pose_names"]
POSE_TO_ID   = ckpt["pose_to_id"]
JOINT_NAMES  = ckpt["joint_names"]
FEAT_MEAN    = np.array(ckpt["feat_mean"], dtype=np.float32)
FEAT_STD     = np.array(ckpt["feat_std"],  dtype=np.float32)

class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim, hidden, layers, n_joints, n_poses):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)
        self.cond_matrix   = nn.Parameter(torch.zeros(n_poses, n_joints))
    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))
        g = self.pool(h).squeeze(-1)
        qs = self.quality_head(g)
        ps = self.pose_head(g)
        fr = self.feedback_head(g)
        pp = F.softmax(ps, dim=1)
        fb = F.relu(fr + pp @ self.cond_matrix)
        return qs, ps, fb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YogaTCN(FEAT_DIM, HIDDEN, LAYERS, N_JOINTS, N_POSES).to(device)
model.load_state_dict(ckpt["model"])
print(f"Pretrain weights loaded on {device}")

# Apply unfreeze policy
for p in model.parameters():
    p.requires_grad = False

# Unfreeze last N TCN blocks
for i in range(LAYERS - N_UNFROZEN_TCN_BLOCKS, LAYERS):
    for p in model.tcn[i].parameters():
        p.requires_grad = True

# Unfreeze all heads + cond_matrix
for p in model.quality_head.parameters():   p.requires_grad = True
for p in model.pose_head.parameters():      p.requires_grad = True
for p in model.feedback_head.parameters():  p.requires_grad = True
model.cond_matrix.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Unfrozen TCN blocks: {list(range(LAYERS-N_UNFROZEN_TCN_BLOCKS, LAYERS))} "
      f"(of {LAYERS})")
print(f"Trainable parameters: {n_trainable:,}  (frozen: {n_frozen:,})")


In [ ]:
LS, RS = 11, 12; LH, RH = 23, 24
LE, RE = 13, 14; LW, RW = 15, 16
LK, RK = 25, 26; LA, RA = 27, 28
VIS_THRESH = 0.3

CHAIN_DOWNSTREAM = {
    "left_elbow":  [15, 17, 19, 21],
    "right_elbow": [16, 18, 20, 22],
    "left_knee":   [27, 29, 31],
    "right_knee":  [28, 30, 32],
    "left_hip":    [25, 27, 29, 31],
    "right_hip":   [26, 28, 30, 32],
}
JOINT_VERTEX = {
    "left_elbow":  LE, "right_elbow": RE,
    "left_knee":   LK, "right_knee":  RK,
    "left_hip":    LH, "right_hip":   RH,
}

def normalize_clip(clip):
    out = clip.copy()
    for t in range(len(out)):
        hip = (out[t, LH, :3] + out[t, RH, :3]) / 2.0
        out[t, :, :3] -= hip
        scale = np.linalg.norm(out[t, LS, :3] - out[t, RS, :3]) + 1e-6
        out[t, :, :3] /= scale
    return out

def _angle_3pts_2d(a, b, c):
    ba = a - b; bc = c - b
    denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    return float(np.degrees(np.arccos(np.clip(np.dot(ba, bc)/denom, -1.0, 1.0))))

def compute_angles(kp):
    xy, vis = kp[:, :2], kp[:, 3]
    def safe(a, b, c):
        return (_angle_3pts_2d(xy[a], xy[b], xy[c])
                if min(vis[a], vis[b], vis[c]) >= VIS_THRESH else 0.0)
    l_el = safe(LS, LE, LW);  r_el = safe(RS, RE, RW)
    l_kn = safe(LH, LK, LA);  r_kn = safe(RH, RK, RA)
    l_hp = safe(LS, LH, LK);  r_hp = safe(RS, RH, RK)
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el-r_el, l_kn-r_kn, l_hp-r_hp,
                     (l_el+r_el)/2, (l_kn+r_kn)/2, (l_hp+r_hp)/2],
                    dtype=np.float32)

def clip_to_features(clip_norm):
    T = len(clip_norm)
    xy = clip_norm[:, :, :2].reshape(T, -1)
    z  = clip_norm[:, :, 2].reshape(T, -1)
    vis = clip_norm[:, :, 3].reshape(T, -1)
    vxy = np.zeros_like(xy); vxy[1:] = xy[1:] - xy[:-1]
    ang = np.stack([compute_angles(clip_norm[t]) for t in range(T)])
    return np.concatenate([xy, vxy, z, vis, ang], axis=1).astype(np.float32)

def _rotate_chain_2d(clip, vertex_idx, downstream_ids, angle_deg):
    theta = np.deg2rad(angle_deg)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    for t in range(len(clip)):
        vx, vy = clip[t, vertex_idx, 0], clip[t, vertex_idx, 1]
        for lm_id in downstream_ids:
            x, y = clip[t, lm_id, 0], clip[t, lm_id, 1]
            dx, dy = x - vx, y - vy
            clip[t, lm_id, 0] = vx + cos_t * dx - sin_t * dy
            clip[t, lm_id, 1] = vy + sin_t * dx + cos_t * dy
    return clip

def corrupt_clip(clip_norm, rng):
    out = clip_norm.copy()
    n_corrupt = rng.integers(N_CORRUPT_JOINTS_MIN, N_CORRUPT_JOINTS_MAX + 1)
    joints = rng.choice(N_JOINTS, size=n_corrupt, replace=False)
    feedback = np.zeros(N_JOINTS, dtype=np.float32)
    for j in joints:
        name  = JOINT_NAMES[j]
        mag   = rng.uniform(CORRUPT_ANGLE_MIN_DEG, CORRUPT_ANGLE_MAX_DEG)
        sign  = rng.choice([-1.0, 1.0])
        _rotate_chain_2d(out, JOINT_VERTEX[name], CHAIN_DOWNSTREAM[name], sign * mag)
        feedback[j] = mag
    jitter = rng.normal(0.0, JITTER_SIGMA, size=out[..., :2].shape).astype(np.float32)
    out[..., :2] += jitter
    return out, feedback

def clean_feedback():
    return np.zeros(N_JOINTS, dtype=np.float32)

print("Feature + corruption helpers ready")


In [ ]:
YOGNET_META = pd.read_csv("path/to/working/extracted_yognet/metadata.csv")
YOGNET_META = YOGNET_META[YOGNET_META["frames_kept"] >= MIN_FRAMES].reset_index(drop=True)
print(f"YogNet clips after short-drop: {len(YOGNET_META)}")

LABEL_CORRECT, LABEL_INCORRECT = 0, 1

all_X, all_q, all_p, all_fb, all_meta = [], [], [], [], []
skipped_no_windows = 0

for _, row in YOGNET_META.iterrows():
    pose_name = row["pose"]
    if pose_name not in POSE_TO_ID:
        continue
    pose_id = POSE_TO_ID[pose_name]

    clip_raw = np.load(row["clip_path"])
    clip_norm = normalize_clip(clip_raw)
    feat_clean = clip_to_features(clip_norm)
    clip_corrupt, fb_vec = corrupt_clip(clip_norm, RNG)
    feat_corrupt = clip_to_features(clip_corrupt)

    T = feat_clean.shape[0]
    starts = list(range(0, T - WINDOW_SIZE + 1, STRIDE))
    if not starts:
        skipped_no_windows += 1
        continue

    detected = (feat_clean[:, 132:165].max(axis=1) > 0)

    for start in starts:
        end = start + WINDOW_SIZE
        if detected[start:end].mean() < MIN_VIS_FRAC:
            continue
        # Clean twin
        all_X.append(feat_clean[start:end])
        all_q.append(LABEL_CORRECT)
        all_p.append(pose_id)
        all_fb.append(clean_feedback())
        all_meta.append({"clip_index": int(row["clip_index"]),
                         "pose": pose_name, "quality_label": "correct",
                         "window_start": start, "dataset": "yognet"})
        # Corrupted twin
        all_X.append(feat_corrupt[start:end])
        all_q.append(LABEL_INCORRECT)
        all_p.append(pose_id)
        all_fb.append(fb_vec)
        all_meta.append({"clip_index": int(row["clip_index"]),
                         "pose": pose_name, "quality_label": "incorrect",
                         "window_start": start, "dataset": "yognet"})

print(f"Clips skipped (no window): {skipped_no_windows}")
print(f"Total windows: {len(all_X)}  "
      f"(correct={sum(q==0 for q in all_q)}, "
      f"incorrect={sum(q==1 for q in all_q)})")

# Clip-level train/val/test split, stratified on pose
meta_df = pd.DataFrame(all_meta)
clip_ids = meta_df["clip_index"].unique()
clip_pose = {cid: meta_df[meta_df["clip_index"] == cid]["pose"].iloc[0]
             for cid in clip_ids}
clip_pose_labels = [POSE_TO_ID[clip_pose[c]] for c in clip_ids]

train_cids, temp_cids, _, temp_lbls = train_test_split(
    clip_ids, clip_pose_labels, test_size=0.30, random_state=42,
    stratify=clip_pose_labels)
val_cids, test_cids = train_test_split(
    temp_cids, test_size=0.50, random_state=42, stratify=temp_lbls)
train_set, val_set, test_set = set(train_cids), set(val_cids), set(test_cids)

X_train_l, q_train_l, p_train_l, fb_train_l = [], [], [], []
X_val_l,   q_val_l,   p_val_l,   fb_val_l   = [], [], [], []
X_test_l,  q_test_l,  p_test_l,  fb_test_l  = [], [], [], []

for X, q, p, fb, m in zip(all_X, all_q, all_p, all_fb, all_meta):
    cid = m["clip_index"]
    if cid in train_set:
        X_train_l.append(X); q_train_l.append(q); p_train_l.append(p); fb_train_l.append(fb)
    elif cid in val_set:
        X_val_l.append(X);   q_val_l.append(q);   p_val_l.append(p);   fb_val_l.append(fb)
    else:
        X_test_l.append(X);  q_test_l.append(q);  p_test_l.append(p);  fb_test_l.append(fb)

X_train_ft = np.stack(X_train_l).astype(np.float32)
X_val_ft   = np.stack(X_val_l).astype(np.float32)
X_test_ft  = np.stack(X_test_l).astype(np.float32)
y_q_train_ft = np.array(q_train_l, dtype=np.int64)
y_q_val_ft   = np.array(q_val_l,   dtype=np.int64)
y_q_test_ft  = np.array(q_test_l,  dtype=np.int64)
y_p_train_ft = np.array(p_train_l, dtype=np.int64)
y_p_val_ft   = np.array(p_val_l,   dtype=np.int64)
y_p_test_ft  = np.array(p_test_l,  dtype=np.int64)
y_fb_train_ft = np.stack(fb_train_l).astype(np.float32)
y_fb_val_ft   = np.stack(fb_val_l).astype(np.float32)
y_fb_test_ft  = np.stack(fb_test_l).astype(np.float32)

print(f"\nTrain: {X_train_ft.shape} | Val: {X_val_ft.shape} | Test: {X_test_ft.shape}")
print(f"Train pose distribution:")
for pid, pname in enumerate(POSE_NAMES):
    print(f"  [{pid}] {pname:22s} {(y_p_train_ft==pid).sum()}")


In [ ]:
class YogaWindowDataset(Dataset):
    def __init__(self, X, y_q, y_p, y_fb, mean, std):
        X_std = (X - mean[None, None, :]) / std[None, None, :]
        self.X    = torch.from_numpy(X_std).float()
        self.y_q  = torch.from_numpy(y_q).long()
        self.y_p  = torch.from_numpy(y_p).long()
        self.y_fb = torch.from_numpy(y_fb).float()
    def __len__(self):  return len(self.y_q)
    def __getitem__(self, i):
        return self.X[i], self.y_q[i], self.y_p[i], self.y_fb[i]

train_ds = YogaWindowDataset(X_train_ft, y_q_train_ft, y_p_train_ft, y_fb_train_ft,
                             FEAT_MEAN, FEAT_STD)
val_ds   = YogaWindowDataset(X_val_ft,   y_q_val_ft,   y_p_val_ft,   y_fb_val_ft,
                             FEAT_MEAN, FEAT_STD)
test_ds  = YogaWindowDataset(X_test_ft,  y_q_test_ft,  y_p_test_ft,  y_fb_test_ft,
                             FEAT_MEAN, FEAT_STD)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

# Optimizer — only unfrozen params
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=FT_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=4, factor=0.5)

criterion_q = nn.CrossEntropyLoss()
criterion_p = nn.CrossEntropyLoss()
print("Dataloaders + optimizer ready")


In [ ]:
def eval_metrics(loader):
    model.eval()
    q_correct = p_correct = total = 0
    fb_sum_sq = 0.0; fb_n = 0
    with torch.no_grad():
        for Xb, yb_q, yb_p, yb_fb in loader:
            Xb = Xb.to(device); yb_q = yb_q.to(device)
            yb_p = yb_p.to(device); yb_fb = yb_fb.to(device)
            logits_q, logits_p, feedback = model(Xb)
            q_correct += (logits_q.argmax(1) == yb_q).sum().item()
            p_correct += (logits_p.argmax(1) == yb_p).sum().item()
            total += yb_q.size(0)
            fb_sum_sq += ((feedback - yb_fb) ** 2).sum().item()
            fb_n += yb_fb.numel()
    return (q_correct / total, p_correct / total,
            (fb_sum_sq / fb_n) ** 0.5)

best_composite = -float("inf")
bad_epochs     = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0; n_seen = 0
    for Xb, yb_q, yb_p, yb_fb in train_loader:
        Xb = Xb.to(device); yb_q = yb_q.to(device)
        yb_p = yb_p.to(device); yb_fb = yb_fb.to(device)

        optimizer.zero_grad()
        logits_q, logits_p, feedback = model(Xb)
        loss_q  = criterion_q(logits_q, yb_q)
        loss_p  = criterion_p(logits_p, yb_p)
        loss_fb = F.smooth_l1_loss(feedback, yb_fb)
        loss    = QUALITY_W_FT * loss_q + POSE_W_FT * loss_p + FB_W_FT * loss_fb
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * yb_q.size(0)
        n_seen       += yb_q.size(0)

    train_loss = running_loss / n_seen
    val_q_acc, val_p_acc, val_fb_rmse = eval_metrics(val_loader)
    # Composite: quality + pose, with feedback RMSE contributing as a penalty
    composite = val_q_acc + val_p_acc
    scheduler.step(composite)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | "
          f"val_q={val_q_acc:.4f} val_p={val_p_acc:.4f} "
          f"val_fb_rmse={val_fb_rmse:.2f} | composite={composite:.4f} "
          f"lr={current_lr:.1e}")

    if composite > best_composite:
        best_composite = composite
        bad_epochs = 0
        torch.save(model.state_dict(), CKPT_BEST_FT)
        print(f"  ✅ saved best checkpoint (q_acc={val_q_acc:.4f}, p_acc={val_p_acc:.4f})")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"  🛑 early stopping after {epoch} epochs")
            break

print(f"\nBest composite val score: {best_composite:.4f}")


In [ ]:
model.load_state_dict(torch.load(CKPT_BEST_FT, map_location=device))
model.eval()

all_q_p, all_q_t   = [], []
all_p_p, all_p_t   = [], []
all_fb_p, all_fb_t = [], []
with torch.no_grad():
    for Xb, yb_q, yb_p, yb_fb in test_loader:
        Xb = Xb.to(device)
        logits_q, logits_p, feedback = model(Xb)
        all_q_p.append(logits_q.argmax(1).cpu().numpy()); all_q_t.append(yb_q.numpy())
        all_p_p.append(logits_p.argmax(1).cpu().numpy()); all_p_t.append(yb_p.numpy())
        all_fb_p.append(feedback.cpu().numpy());          all_fb_t.append(yb_fb.numpy())

all_q_p = np.concatenate(all_q_p); all_q_t = np.concatenate(all_q_t)
all_p_p = np.concatenate(all_p_p); all_p_t = np.concatenate(all_p_t)
all_fb_p = np.concatenate(all_fb_p); all_fb_t = np.concatenate(all_fb_t)

print("=" * 60)
print("QUALITY HEAD (YogNet test)")
print(f"Accuracy: {(all_q_p == all_q_t).mean():.4f}")
print(confusion_matrix(all_q_t, all_q_p))
print(classification_report(all_q_t, all_q_p,
                            target_names=["correct","incorrect"], digits=3))

print("=" * 60)
print("POSE HEAD (YogNet test, 7 classes)")
print(f"Accuracy: {(all_p_p == all_p_t).mean():.4f}")
print(classification_report(all_p_t, all_p_p,
                            target_names=POSE_NAMES, digits=3))

print("=" * 60)
print("FEEDBACK HEAD — per-joint MAE on corrupted samples (deg)")
corrupted_mask = (all_q_t == 1)
for j, name in enumerate(JOINT_NAMES):
    if corrupted_mask.sum() == 0:
        continue
    mae_corr = np.abs(all_fb_p[corrupted_mask, j]
                      - all_fb_t[corrupted_mask, j]).mean()
    mae_clean = np.abs(all_fb_p[~corrupted_mask, j]).mean()
    print(f"  {name:20s} MAE(corrupted)={mae_corr:5.2f}  "
          f"MAE(clean baseline)={mae_clean:5.2f}")

# Pose-angle + landmark means on YogNet clean train windows (for Stage 7 inference)
ANGLE_START, ANGLE_COLS = 198, 6
clean_mask = (y_q_train_ft == 0)
pose_angle_means_ft = {}
pose_landmark_means_ft = {}
for pid in range(N_POSES):
    m = (y_p_train_ft == pid) & clean_mask
    if m.sum() > 0:
        ang = X_train_ft[m, :, ANGLE_START:ANGLE_START+ANGLE_COLS].mean(axis=(0,1))
        pose_angle_means_ft[pid] = ang.astype(np.float32)
        xy = X_train_ft[m, :, 0:66].mean(axis=(0,1)).reshape(33, 2)
        pose_landmark_means_ft[pid] = xy.astype(np.float32)
    else:
        pose_angle_means_ft[pid]    = np.zeros(ANGLE_COLS, dtype=np.float32)
        pose_landmark_means_ft[pid] = np.zeros((33, 2),    dtype=np.float32)

torch.save({
    "model":               model.state_dict(),
    "feat_dim":            FEAT_DIM,
    "hidden":              HIDDEN,
    "layers":              LAYERS,
    "n_joints":            N_JOINTS,
    "n_poses":             N_POSES,
    "joint_names":         JOINT_NAMES,
    "pose_names":          POSE_NAMES,
    "pose_to_id":          POSE_TO_ID,
    "feat_mean":           FEAT_MEAN.tolist(),
    "feat_std":            FEAT_STD.tolist(),
    "pose_angle_means":    {k: v.tolist() for k, v in pose_angle_means_ft.items()},
    "pose_landmark_means": {k: v.tolist() for k, v in pose_landmark_means_ft.items()},
    "max_feedback_deg":    45.0,
    "angle_start":         ANGLE_START,
    "angle_cols":          ANGLE_COLS,
    "window_size":         WINDOW_SIZE,
    "source":              "yognet-finetuned",
}, CKPT_FULL_FT)
print(f"\n✅ Fine-tuned checkpoint saved: {CKPT_FULL_FT}")
print("   Ready for Stage 7 (real-time inference)")


In [ ]:
# Stage 7 — Real-Time Inference with Skeleton Overlay (YogNet-finetuned model)
import os, cv2, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

CKPT_PATH = "path/to/working/yoga_tcn_finetuned.pt"
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

FEAT_DIM      = ckpt["feat_dim"]
HIDDEN        = ckpt["hidden"]
LAYERS        = ckpt["layers"]
N_JOINTS      = ckpt["n_joints"]
N_POSES       = ckpt["n_poses"]
JOINT_NAMES   = ckpt["joint_names"]
POSE_NAMES    = ckpt["pose_names"]
FEAT_MEAN     = np.array(ckpt["feat_mean"],  dtype=np.float32)
FEAT_STD      = np.array(ckpt["feat_std"],   dtype=np.float32)
WINDOW_SIZE   = ckpt["window_size"]
ANGLE_START   = ckpt["angle_start"]
ANGLE_COLS    = ckpt["angle_cols"]
MAX_FB_DEG    = ckpt["max_feedback_deg"]

POSE_ANGLE_MEANS    = {int(k): np.array(v, dtype=np.float32)
                       for k, v in ckpt["pose_angle_means"].items()}
POSE_LANDMARK_MEANS = {int(k): np.array(v, dtype=np.float32).reshape(33, 2)
                       for k, v in ckpt["pose_landmark_means"].items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Checkpoint source: {ckpt.get('source', 'unknown')}")
print(f"Device: {device}")
print(f"Poses ({N_POSES}): {POSE_NAMES}")
print(f"Window size: {WINDOW_SIZE} | Feat dim: {FEAT_DIM}")


In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k, dilation, pdrop=0.2):
        super().__init__()
        pad = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv1 = nn.Conv1d(cin, cout, k, padding=0, dilation=dilation)
        self.pad2  = nn.ConstantPad1d((pad, 0), 0.0)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=0, dilation=dilation)
        self.drop  = nn.Dropout(pdrop)
        self.res   = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x):
        y = F.relu(self.conv1(self.pad1(x)))
        y = self.drop(y)
        y = F.relu(self.conv2(self.pad2(y)))
        y = self.drop(y)
        return y + self.res(x)

class YogaTCN(nn.Module):
    def __init__(self, feat_dim, hidden, layers, n_joints, n_poses):
        super().__init__()
        blocks, cin = [], feat_dim
        for i in range(layers):
            blocks.append(TCNBlock(cin, hidden, k=3, dilation=2**i))
            cin = hidden
        self.tcn  = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.quality_head  = nn.Linear(hidden, 2)
        self.pose_head     = nn.Linear(hidden, n_poses)
        self.feedback_head = nn.Linear(hidden, n_joints)
        self.cond_matrix   = nn.Parameter(torch.zeros(n_poses, n_joints))
    def forward(self, x):
        h = self.tcn(x.transpose(1, 2))
        g = self.pool(h).squeeze(-1)
        qs = self.quality_head(g)
        ps = self.pose_head(g)
        fr = self.feedback_head(g)
        pp = F.softmax(ps, dim=1)
        fb = F.relu(fr + pp @ self.cond_matrix)
        return qs, ps, fb

model_inf = YogaTCN(FEAT_DIM, HIDDEN, LAYERS, N_JOINTS, N_POSES)
model_inf.load_state_dict(ckpt["model"])
model_inf.to(device).eval()
print("Model ready for inference")


In [ ]:
MODEL_PATH = "path/to/working/models/pose_landmarker_full.task"
MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/"
              "pose_landmarker/pose_landmarker_full/float16/latest/"
              "pose_landmarker_full.task")
if not os.path.exists(MODEL_PATH):
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

landmarker_inf = mp_vision.PoseLandmarker.create_from_options(
    mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_poses=1,
    )
)

# Indices mirror Stage 2/6
LS, RS = 11, 12
LE, RE = 13, 14
LW, RW = 15, 16
LH, RH = 23, 24
LK, RK = 25, 26
LA, RA = 27, 28
VIS_THRESH_INF = 0.3

def detect_keypoints_inf(frame_bgr):
    rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = landmarker_inf.detect(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not result.pose_landmarks:
        return None
    lms = result.pose_landmarks[0]
    return np.array([[lm.x, lm.y, lm.z, getattr(lm, "visibility", 1.0)]
                     for lm in lms], dtype=np.float32)

def normalize_frame_inf(kp):
    kp = kp.copy()
    hip = (kp[LH, :3] + kp[RH, :3]) / 2.0
    kp[:, :3] -= hip
    scale = np.linalg.norm(kp[LS, :3] - kp[RS, :3]) + 1e-6
    kp[:, :3] /= scale
    return kp

def _angle_3pts_2d_inf(a, b, c):
    ba = a - b; bc = c - b
    cos = np.clip(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8), -1, 1)
    return float(np.degrees(np.arccos(cos)))

def compute_angles_inf(kp):
    xy, vis = kp[:, :2], kp[:, 3]
    def safe(a, b, c):
        return (_angle_3pts_2d_inf(xy[a], xy[b], xy[c])
                if min(vis[a], vis[b], vis[c]) >= VIS_THRESH_INF else 0.0)
    l_el = safe(LS, LE, LW);  r_el = safe(RS, RE, RW)
    l_kn = safe(LH, LK, LA);  r_kn = safe(RH, RK, RA)
    l_hp = safe(LS, LH, LK);  r_hp = safe(RS, RH, RK)
    return np.array([l_el, r_el, l_kn, r_kn, l_hp, r_hp,
                     l_el-r_el, l_kn-r_kn, l_hp-r_hp,
                     (l_el+r_el)/2, (l_kn+r_kn)/2, (l_hp+r_hp)/2], dtype=np.float32)

def build_feature_window_inf(kp_buffer):
    clip = np.stack(kp_buffer, axis=0)
    norm = np.stack([normalize_frame_inf(clip[t]) for t in range(len(clip))])
    T = len(norm)
    xy  = norm[:, :, :2].reshape(T, -1)
    z   = norm[:, :, 2].reshape(T, -1)
    vis = norm[:, :, 3].reshape(T, -1)
    vxy = np.zeros_like(xy); vxy[1:] = xy[1:] - xy[:-1]
    ang = np.stack([compute_angles_inf(norm[t]) for t in range(T)])
    feat = np.concatenate([xy, vxy, z, vis, ang], axis=1).astype(np.float32)
    return (feat - FEAT_MEAN[None]) / FEAT_STD[None]

print("MediaPipe + feature helpers ready")


In [ ]:
# Severity thresholds (degrees)
THRESH_GREEN  = 10.0
THRESH_YELLOW = 25.0
CONF_GATE     = 0.6   # pose confidence below this -> no ideal overlay

# BGR colors
COL_GREEN  = (50,  205,  50)
COL_YELLOW = (0,   215, 255)
COL_RED    = (0,    60, 220)
COL_GHOST  = (200, 200, 200)
COL_PANEL  = (20,   20,  20)

JOINT_VERTEX_IDS = [13, 14, 25, 26, 23, 24]   # matches JOINT_NAMES order
FEEDBACK_LM_IDS  = {11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28}

IDEAL_CONNECTIONS = [
    (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (24, 26), (26, 28),
]

LM_TO_JOINT = {
    11: "left_hip",    12: "right_hip",
    13: "left_elbow",  14: "right_elbow",
    15: "left_elbow",  16: "right_elbow",
    23: "left_hip",    24: "right_hip",
    25: "left_knee",   26: "right_knee",
    27: "left_knee",   28: "right_knee",
}

def severity_color(deg):
    if deg < THRESH_GREEN:  return COL_GREEN
    if deg < THRESH_YELLOW: return COL_YELLOW
    return COL_RED

def bone_color(lm_a, lm_b, sev):
    da = sev.get(LM_TO_JOINT.get(lm_a, ""), 0.0)
    db = sev.get(LM_TO_JOINT.get(lm_b, ""), 0.0)
    return severity_color(max(da, db))

print("Rendering constants ready")


In [ ]:
def get_ideal_pixel_coords(kp_raw, pose_id, frame_h, frame_w):
    ref = POSE_LANDMARK_MEANS.get(pose_id)
    if ref is None:
        return None
    hip_px = ((kp_raw[LH, 0] + kp_raw[RH, 0]) / 2) * frame_w
    hip_py = ((kp_raw[LH, 1] + kp_raw[RH, 1]) / 2) * frame_h
    sh_dx  = (kp_raw[LS, 0] - kp_raw[RS, 0]) * frame_w
    sh_dy  = (kp_raw[LS, 1] - kp_raw[RS, 1]) * frame_h
    scale  = np.sqrt(sh_dx**2 + sh_dy**2) + 1e-6
    return {lm_id: (int(hip_px + ref[lm_id, 0] * scale),
                    int(hip_py + ref[lm_id, 1] * scale))
            for lm_id in FEEDBACK_LM_IDS}

def draw_detected_skeleton(frame, kp_raw, sev, H, W):
    for a, b in IDEAL_CONNECTIONS:
        xa = int(kp_raw[a, 0] * W); ya = int(kp_raw[a, 1] * H)
        xb = int(kp_raw[b, 0] * W); yb = int(kp_raw[b, 1] * H)
        cv2.line(frame, (xa, ya), (xb, yb), bone_color(a, b, sev), 2)
    for lm_id in FEEDBACK_LM_IDS:
        px = int(kp_raw[lm_id, 0] * W); py = int(kp_raw[lm_id, 1] * H)
        joint_name = LM_TO_JOINT.get(lm_id, "")
        col = severity_color(sev.get(joint_name, 0.0))
        r = 7 if lm_id in set(JOINT_VERTEX_IDS) else 4
        cv2.circle(frame, (px, py), r, col, -1)

def draw_ideal_skeleton(frame, kp_raw, pose_id, pose_conf, H, W):
    coords = get_ideal_pixel_coords(kp_raw, pose_id, H, W)
    if coords is None:
        return
    overlay = frame.copy()
    for a, b in IDEAL_CONNECTIONS:
        if a in coords and b in coords:
            cv2.line(overlay, coords[a], coords[b], COL_GHOST, 2)
    for lm_id in FEEDBACK_LM_IDS:
        if lm_id in coords:
            r = 6 if lm_id in set(JOINT_VERTEX_IDS) else 3
            cv2.circle(overlay, coords[lm_id], r, COL_GHOST, -1)
    cv2.addWeighted(overlay, pose_conf, frame, 1.0 - pose_conf, 0, frame)

def draw_text_panel(frame, result):
    sev = result["severities"]
    lines = [
        f"Pose : {result['pose_name']} ({result['pose_conf']:.0%})",
        f"Quality: {result['quality_score']:.2f}  (1=ideal)",
        f"Mean dev: {result['mean_dev']:.1f} deg",
    ]
    for name in sorted(sev, key=lambda k: -sev[k]):
        deg = sev[name]
        if deg >= THRESH_GREEN:
            flag = "!!" if deg >= THRESH_YELLOW else " !"
            lines.append(f"  {flag} {name}: {deg:.1f}\u00b0")

    panel_h = len(lines) * 22 + 12
    cv2.rectangle(frame, (5, 5), (290, panel_h), COL_PANEL, -1)
    cv2.rectangle(frame, (5, 5), (290, panel_h), (80, 80, 80), 1)
    for i, line in enumerate(lines):
        cv2.putText(frame, line, (10, 24 + i * 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (240, 240, 240), 1, cv2.LINE_AA)

print("Draw functions ready")


In [ ]:
INFER_STRIDE = 8   # run TCN every N frames

def run_yoga_inference(video_in, video_out):
    cap = cv2.VideoCapture(video_in)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_in}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (W, H))

    kp_buffer   = []
    last_result = None
    last_kp_raw = None
    frame_idx   = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        kp_raw         = detect_keypoints_inf(frame)
        detected_frame = kp_raw is not None

        kp_buffer.append(kp_raw if detected_frame
                         else np.zeros((33, 4), dtype=np.float32))
        if len(kp_buffer) > WINDOW_SIZE:
            kp_buffer.pop(0)
        if detected_frame:
            last_kp_raw = kp_raw

        # Run TCN every INFER_STRIDE frames once buffer is full
        if len(kp_buffer) == WINDOW_SIZE and frame_idx % INFER_STRIDE == 0:
            feat = build_feature_window_inf(kp_buffer)
            xt = torch.from_numpy(feat[None]).to(device)
            with torch.no_grad():
                logits_q, logits_p, feedback = model_inf(xt)
            q_probs = torch.softmax(logits_q, dim=1).cpu().numpy()[0]
            p_probs = torch.softmax(logits_p, dim=1).cpu().numpy()[0]
            fb_deg  = feedback.cpu().numpy()[0]
            pose_id = int(np.argmax(p_probs))
            last_result = {
                "pose_id":       pose_id,
                "pose_name":     POSE_NAMES[pose_id],
                "pose_conf":     float(p_probs[pose_id]),
                "quality_score": float(q_probs[0]),    # P(correct)
                "mean_dev":      float(fb_deg.mean()),
                "severities":    {JOINT_NAMES[j]: float(fb_deg[j])
                                  for j in range(N_JOINTS)},
            }

        render = frame.copy()
        if last_result is not None:
            if detected_frame and last_kp_raw is not None:
                draw_detected_skeleton(render, last_kp_raw,
                                       last_result["severities"], H, W)
            if (detected_frame
                    and last_kp_raw is not None
                    and last_result["pose_conf"] >= CONF_GATE):
                draw_ideal_skeleton(render, last_kp_raw,
                                    last_result["pose_id"],
                                    last_result["pose_conf"], H, W)
            draw_text_panel(render, last_result)

        writer.write(render)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"Done. Wrote {frame_idx} frames -> {video_out}")

print("run_yoga_inference ready")


In [ ]:
# Pick any Yoga clip. For live webcam, swap cv2.VideoCapture(VIDEO_IN) -> cv2.VideoCapture(0).


In [ ]:

VIDEO_IN  = "path/to/yognet7poses/Shavasana/13.mp4"
VIDEO_OUT = "path/to/working/Shavasana13_output.mp4"

run_yoga_inference(VIDEO_IN, VIDEO_OUT)


In [ ]:

VIDEO_IN  = "path/to/yognet7poses/Shavasana/55.mp4"
VIDEO_OUT = "path/to/working/Shavasana55_output.mp4"

run_yoga_inference(VIDEO_IN, VIDEO_OUT)


In [ ]:

VIDEO_IN  = "path/to/yognet7poses/Shavasana/56.mp4"
VIDEO_OUT = "path/to/working/Shavasana56_output.mp4"

run_yoga_inference(VIDEO_IN, VIDEO_OUT)


In [ ]:
VIDEO_IN  = "path/to/yognet7poses/Shavasana/13.mp4"
VIDEO_OUT = "path/to/working/Shavasana13_output.mp4"

# 1. Does the input exist and open?
print(f"Input exists: {os.path.exists(VIDEO_IN)}")
if os.path.exists(VIDEO_IN):
    print(f"Input size  : {os.path.getsize(VIDEO_IN)/1024:.1f} KB")
    cap = cv2.VideoCapture(VIDEO_IN)
    print(f"Input opens : {cap.isOpened()}")
    if cap.isOpened():
        print(f"  FPS       : {cap.get(cv2.CAP_PROP_FPS)}")
        print(f"  Frames    : {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
        print(f"  Dimensions: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
    cap.release()

print("\npath/to/working contents:")
for f in sorted(os.listdir("path/to/working")):
    full = f"path/to/working/{f}"
    if os.path.isfile(full):
        print(f"  {f}  ({os.path.getsize(full)/1024:.1f} KB)")


In [ ]:
# Wrap-up — Bundle all project artifacts for download
import os, shutil, json, zipfile, pathlib, datetime, glob
import pandas as pd

RESULTS = "path/to/working/results"
shutil.rmtree(RESULTS, ignore_errors=True)
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(f"{RESULTS}/models",        exist_ok=True)
os.makedirs(f"{RESULTS}/metadata",      exist_ok=True)
os.makedirs(f"{RESULTS}/windows_stats", exist_ok=True)
os.makedirs(f"{RESULTS}/sample_output", exist_ok=True)

def _copy_if_exists(src, dst):
    if os.path.exists(src):
        shutil.copy2(src, dst)
        return True
    return False

# --- Trained checkpoints ---
checkpoint_files = [
    ("path/to/working/yoga_tcn_pretrain.pt",         "yoga_tcn_pretrain.pt"),
    ("path/to/working/yoga_tcn_pretrain_best.pt",    "yoga_tcn_pretrain_best.pt"),
    ("path/to/working/yoga_tcn_finetuned.pt",        "yoga_tcn_finetuned.pt"),
    ("path/to/working/yoga_tcn_finetune_best.pt",    "yoga_tcn_finetune_best.pt"),
]
for src, dstname in checkpoint_files:
    if _copy_if_exists(src, f"{RESULTS}/models/{dstname}"):
        size_mb = os.path.getsize(f"{RESULTS}/models/{dstname}") / 1024 / 1024
        print(f"  ✓ {dstname}  ({size_mb:.1f} MB)")
    else:
        print(f"  — {dstname} not found, skipping")

# --- Dataset metadata CSVs ---
metadata_files = [
    ("path/to/working/extracted_3dyoga90/metadata.csv", "3dyoga90_metadata.csv"),
    ("path/to/working/extracted_yognet/metadata.csv",   "yognet_metadata.csv"),
    ("path/to/working/windows/meta_train.csv",          "pretrain_windows_train.csv"),
    ("path/to/working/windows/meta_val.csv",            "pretrain_windows_val.csv"),
    ("path/to/working/windows/meta_test.csv",           "pretrain_windows_test.csv"),
]
for src, dstname in metadata_files:
    _copy_if_exists(src, f"{RESULTS}/metadata/{dstname}")

# --- Feature standardization stats (important for anyone re-running inference) ---
_copy_if_exists("path/to/working/windows/feature_stats.npz",
                f"{RESULTS}/windows_stats/feature_stats.npz")

# --- Sample inference output if it exists ---
_copy_if_exists("path/to/working/yoga_output.mp4",
                f"{RESULTS}/sample_output/yoga_output.mp4")

# --- Results summary (copy the key numbers you'd want to read later) ---
summary = {
    "project":    "Yoga Pose Quality Assessment (CS7150)",
    "timestamp":  datetime.datetime.now().isoformat(),
    "target_poses": POSE_NAMES,
    "datasets": {
        "pretrain_3dyoga90": {
            "source":    "https://github.com/seonokkim/3DYoga90",
            "format":    "ISLR-schema parquets (MediaPipe Holistic 543-point)",
            "clips_extracted": 851,
            "clip_counts_per_pose": {
                "cobra": 128, "corpse": 95, "downward_facing_dog": 245,
                "mountain": 163, "tree": 69, "triangle": 107, "warrior": 44,
            },
            "notes": "warrior-1 only (warrior-2/3 excluded); detection rate "
                     "100% (ISLR parquets are pre-clean).",
        },
        "finetune_yognet": {
            "source":   "YAR — Yoga Asanas Recognition (Yadav et al. 2022)",
            "format":   "RGB videos, 4 view angles per pose",
            "clips_extracted": 434,
            "clip_counts_per_pose": {
                "cobra": 64, "corpse": 63, "downward_facing_dog": 64,
                "mountain": 64, "tree": 63, "triangle": 53, "warrior": 63,
            },
            "frame_stride": 2,
            "notes": "corpse has bimodal detection — 60% of videos ≥90% "
                     "detection, 16% below 20% due to BlazePose limitations "
                     "on supine poses.",
        },
    },
    "augmentation": {
        "scheme": "clip-level targeted-joint chain rotation + global jitter",
        "corruption_joints": "1-2 of 6 feedback joints (elbows, knees, hips)",
        "rotation_range_deg": [0.0, 45.0],
        "jitter_sigma": 0.01,
        "chain_rotation": "rotates full downstream kinematic chain consistently "
                          "across all frames of a clip (e.g., elbow corruption "
                          "rotates wrist + fingers together)",
        "mix_ratio": "1:1 clean:corrupted per pose",
    },
    "model": {
        "architecture": "Causal TCN (6 blocks, dilation 2^i) + 3 heads + "
                        "soft pose-conditioning matrix",
        "hidden_dim":   128,
        "feature_dim":  210,
        "window_size":  32,
        "stride":       32,
        "n_poses":      7,
        "n_feedback_joints": 6,
    },
    "results": {
        "stage_3_pretrain_3dyoga90_test": {
            "quality_acc":    0.9977,
            "pose_acc":       0.9475,
            "feedback_mae_per_joint_deg": {
                "left_elbow": 3.23, "right_elbow": 2.49, "left_knee": 2.72,
                "right_knee": 3.52, "left_hip": 3.07, "right_hip": 3.39,
            },
            "note": "quality accuracy saturated — likely shortcut on synthetic "
                    "corruption signature.",
        },
        "stage_5_pretrain_on_yognet_no_finetune": {
            "pose_acc":       0.4561,
            "quality_p_correct_mean": 0.578,
            "note": "confirmed domain shift — YogNet's 4-view protocol exposed "
                    "pretrain's canonical-view specialization.",
        },
        "stage_6_finetune_yognet_test": {
            "quality_acc":    0.9150,
            "pose_acc":       0.9631,
            "feedback_mae_corrupted_per_joint_deg": {
                "left_elbow": 5.44, "right_elbow": 7.68, "left_knee": 4.81,
                "right_knee": 7.55, "left_hip": 5.71, "right_hip": 5.28,
            },
            "feedback_mae_clean_per_joint_deg": {
                j: 0.0 for j in JOINT_NAMES
            },
            "quality_confusion_matrix": {
                "correct_predicted_correct":   426,
                "correct_predicted_incorrect": 62,
                "incorrect_predicted_correct": 21,
                "incorrect_predicted_incorrect": 467,
            },
            "note": "unfroze last 2 TCN blocks + all heads + cond_matrix; "
                    "LR 3e-4; 1:1 YogNet synthetic corruption.",
        },
    },
    "deliverables": {
        "primary_checkpoint": "models/yoga_tcn_finetuned.pt",
        "pretrain_checkpoint": "models/yoga_tcn_pretrain.pt",
        "feature_stats":       "windows_stats/feature_stats.npz",
        "sample_inference":    "sample_output/yoga_output.mp4",
    },
}
with open(f"{RESULTS}/results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# --- Human-readable README ---
readme = f"""# Yoga Pose Quality Assessment — Results Bundle

Target poses: {', '.join(POSE_NAMES)}

## Directory structure

- `models/` — PyTorch checkpoints
  - `yoga_tcn_finetuned.pt` ← **primary deliverable**, use for inference
  - `yoga_tcn_pretrain.pt` — 3DYoga90-only pretrain (no YogNet fine-tune)
  - `*_best.pt` — best-val-score checkpoints from training
- `metadata/` — per-dataset clip manifests
- `windows_stats/feature_stats.npz` — train-set FEAT_MEAN / FEAT_STD (210-dim)
- `sample_output/` — rendered inference example
- `results_summary.json` — all training/test metrics

## Checkpoint contents (either checkpoint unpacks the same way)

```python
import torch
ckpt = torch.load("models/yoga_tcn_finetuned.pt", weights_only=False)
# Keys: model, feat_dim, hidden, layers, n_joints, n_poses,
#       joint_names, pose_names, pose_to_id,
#       feat_mean, feat_std, pose_angle_means, pose_landmark_means,
#       max_feedback_deg, angle_start, angle_cols, window_size
```

## Final metrics (Stage 6, YogNet test set)

- Quality accuracy: **91.5%**
- Pose accuracy (7-way): **96.3%**
- Feedback MAE on corrupted samples: 4.8-7.7° per joint
- Feedback MAE on clean samples: 0.0° (ReLU-gated — "no correction needed")

## Stage-by-stage pipeline

1. 3DYoga90 parquet ETL → 851 clips (7 poses, warrior-1 only)
2. Feature engineering + 1:1 synthetic-incorrect augmentation → 11,204 windows
3. TCN pretrain on 3DYoga90 → `yoga_tcn_pretrain.pt`
4. YogNet video extraction (MediaPipe, stride=2) → 434 clips
5. Cross-dataset test (diagnostic, no training) → pose acc 46% (big drop)
6. Fine-tune on YogNet with 1:1 synthetic corruption → `yoga_tcn_finetuned.pt`
7. Real-time inference with skeleton overlay

Generated: {summary["timestamp"]}
"""
with open(f"{RESULTS}/README.md", "w") as f:
    f.write(readme)

# --- Zip the whole bundle for one-click download ---
zip_path = "path/to/working/cs7150_yoga_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RESULTS):
        for fn in files:
            full = os.path.join(root, fn)
            rel  = os.path.relpath(full, RESULTS)
            zf.write(full, f"cs7150_yoga_results/{rel}")

zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"\n{'=' * 60}")
print(f"BUNDLE COMPLETE")
print(f"{'=' * 60}")
print(f"Unzipped results dir : {RESULTS}")
print(f"Zipped download      : {zip_path}  ({zip_size_mb:.1f} MB)")
print(f"\nDownloadable from Kaggle's right-side Output panel:")
print(f"  - {zip_path}                (full bundle)")
print(f"  - {RESULTS}/README.md       (summary)")
print(f"  - {RESULTS}/results_summary.json")
print(f"  - {RESULTS}/models/yoga_tcn_finetuned.pt  (primary model)")

# List contents as a final sanity check
print(f"\nBundle contents:")
for root, _, files in os.walk(RESULTS):
    for fn in sorted(files):
        full = os.path.join(root, fn)
        rel = os.path.relpath(full, RESULTS)
        size_kb = os.path.getsize(full) / 1024
        print(f"  {rel:60s} ({size_kb:7.1f} KB)")


---
# Results Visualization

In [ ]:
# Parse training logs into structured data for plotting
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Stage 3 log (3DYoga90 pretrain) ---
STAGE3_LOG = """Epoch 001 | loss=2.0505 | val_quality_acc=0.5466
Epoch 002 | loss=1.5614 | val_quality_acc=0.9911
Epoch 003 | loss=1.2586 | val_quality_acc=0.9981
Epoch 004 | loss=1.2564 | val_quality_acc=0.9987
Epoch 005 | loss=1.2010 | val_quality_acc=0.9968
Epoch 006 | loss=1.1857 | val_quality_acc=0.9994
Epoch 007 | loss=1.1850 | val_quality_acc=0.9981
Epoch 008 | loss=1.1367 | val_quality_acc=0.9987
Epoch 009 | loss=1.1275 | val_quality_acc=0.9987
Epoch 010 | loss=1.0966 | val_quality_acc=1.0000
Epoch 011 | loss=1.0694 | val_quality_acc=0.9994
Epoch 012 | loss=1.0439 | val_quality_acc=0.9987
Epoch 013 | loss=1.0224 | val_quality_acc=0.9981
Epoch 014 | loss=1.0122 | val_quality_acc=0.9879
Epoch 015 | loss=0.9717 | val_quality_acc=0.9981
Epoch 016 | loss=0.9733 | val_quality_acc=0.9981
Epoch 017 | loss=0.9076 | val_quality_acc=0.9987
Epoch 018 | loss=0.8835 | val_quality_acc=0.9987
Epoch 019 | loss=0.8709 | val_quality_acc=1.0000
Epoch 020 | loss=0.8610 | val_quality_acc=1.0000"""

# --- Stage 6 log (YogNet fine-tune) ---
STAGE6_LOG = """Epoch 001 | loss=5.9089 | val_q=0.8510 val_p=0.6185 val_fb_rmse=8.15 | composite=1.4695 lr=3.0e-04
Epoch 002 | loss=2.5893 | val_q=0.8804 val_p=0.8668 val_fb_rmse=8.23 | composite=1.7472 lr=3.0e-04
Epoch 003 | loss=1.9793 | val_q=0.8747 val_p=0.9289 val_fb_rmse=8.23 | composite=1.8036 lr=3.0e-04
Epoch 004 | loss=1.8093 | val_q=0.8916 val_p=0.9244 val_fb_rmse=8.23 | composite=1.8160 lr=3.0e-04
Epoch 005 | loss=1.7398 | val_q=0.8962 val_p=0.9368 val_fb_rmse=8.23 | composite=1.8330 lr=3.0e-04
Epoch 006 | loss=1.6855 | val_q=0.9052 val_p=0.9424 val_fb_rmse=8.23 | composite=1.8476 lr=3.0e-04
Epoch 007 | loss=1.6406 | val_q=0.9007 val_p=0.9458 val_fb_rmse=8.23 | composite=1.8465 lr=3.0e-04
Epoch 008 | loss=1.6283 | val_q=0.9108 val_p=0.9537 val_fb_rmse=8.23 | composite=1.8646 lr=3.0e-04
Epoch 009 | loss=1.6158 | val_q=0.8950 val_p=0.9515 val_fb_rmse=8.23 | composite=1.8465 lr=3.0e-04
Epoch 010 | loss=1.5846 | val_q=0.9063 val_p=0.9526 val_fb_rmse=8.23 | composite=1.8589 lr=3.0e-04
Epoch 011 | loss=1.5755 | val_q=0.9120 val_p=0.9503 val_fb_rmse=8.22 | composite=1.8623 lr=3.0e-04
Epoch 012 | loss=1.5591 | val_q=0.9074 val_p=0.9537 val_fb_rmse=8.22 | composite=1.8612 lr=3.0e-04
Epoch 013 | loss=1.5606 | val_q=0.9142 val_p=0.9526 val_fb_rmse=8.21 | composite=1.8668 lr=3.0e-04
Epoch 014 | loss=1.5518 | val_q=0.9063 val_p=0.9492 val_fb_rmse=8.21 | composite=1.8555 lr=3.0e-04
Epoch 015 | loss=1.5504 | val_q=0.9086 val_p=0.9628 val_fb_rmse=8.22 | composite=1.8713 lr=3.0e-04
Epoch 016 | loss=1.5318 | val_q=0.9097 val_p=0.9616 val_fb_rmse=8.21 | composite=1.8713 lr=3.0e-04
Epoch 017 | loss=1.5141 | val_q=0.9210 val_p=0.9594 val_fb_rmse=8.21 | composite=1.8804 lr=3.0e-04
Epoch 018 | loss=1.5344 | val_q=0.9086 val_p=0.9661 val_fb_rmse=8.21 | composite=1.8747 lr=3.0e-04
Epoch 019 | loss=1.5325 | val_q=0.9187 val_p=0.9605 val_fb_rmse=8.21 | composite=1.8792 lr=3.0e-04
Epoch 020 | loss=1.5110 | val_q=0.9165 val_p=0.9492 val_fb_rmse=8.21 | composite=1.8657 lr=3.0e-04
Epoch 021 | loss=1.5175 | val_q=0.9233 val_p=0.9571 val_fb_rmse=8.21 | composite=1.8804 lr=3.0e-04
Epoch 022 | loss=1.5138 | val_q=0.9244 val_p=0.9673 val_fb_rmse=8.21 | composite=1.8916 lr=3.0e-04
Epoch 023 | loss=1.4995 | val_q=0.9221 val_p=0.9503 val_fb_rmse=8.21 | composite=1.8725 lr=3.0e-04
Epoch 024 | loss=1.5139 | val_q=0.9187 val_p=0.9684 val_fb_rmse=8.21 | composite=1.8871 lr=3.0e-04
Epoch 025 | loss=1.5027 | val_q=0.9153 val_p=0.9402 val_fb_rmse=8.21 | composite=1.8555 lr=3.0e-04
Epoch 026 | loss=1.4918 | val_q=0.9187 val_p=0.9639 val_fb_rmse=8.21 | composite=1.8826 lr=3.0e-04
Epoch 027 | loss=1.4894 | val_q=0.9165 val_p=0.9616 val_fb_rmse=8.21 | composite=1.8781 lr=1.5e-04
Epoch 028 | loss=1.4865 | val_q=0.9210 val_p=0.9560 val_fb_rmse=8.21 | composite=1.8770 lr=1.5e-04
Epoch 029 | loss=1.4780 | val_q=0.9312 val_p=0.9616 val_fb_rmse=8.21 | composite=1.8928 lr=1.5e-04
Epoch 030 | loss=1.4756 | val_q=0.9199 val_p=0.9616 val_fb_rmse=8.21 | composite=1.8815 lr=1.5e-04
Epoch 031 | loss=1.4731 | val_q=0.9210 val_p=0.9650 val_fb_rmse=8.21 | composite=1.8860 lr=1.5e-04
Epoch 032 | loss=1.4808 | val_q=0.9221 val_p=0.9673 val_fb_rmse=8.21 | composite=1.8894 lr=1.5e-04
Epoch 033 | loss=1.4721 | val_q=0.9266 val_p=0.9628 val_fb_rmse=8.21 | composite=1.8894 lr=1.5e-04
Epoch 034 | loss=1.4668 | val_q=0.9176 val_p=0.9650 val_fb_rmse=8.21 | composite=1.8826 lr=7.5e-05
Epoch 035 | loss=1.4641 | val_q=0.9221 val_p=0.9650 val_fb_rmse=8.21 | composite=1.8871 lr=7.5e-05
Epoch 036 | loss=1.4639 | val_q=0.9255 val_p=0.9673 val_fb_rmse=8.21 | composite=1.8928 lr=7.5e-05
Epoch 037 | loss=1.4698 | val_q=0.9244 val_p=0.9661 val_fb_rmse=8.21 | composite=1.8905 lr=7.5e-05"""

def parse_stage3(log):
    rows = []
    for line in log.strip().splitlines():
        m = re.match(r"Epoch (\d+) \| loss=([\d.]+) \| val_quality_acc=([\d.]+)", line)
        if m:
            rows.append({"epoch": int(m.group(1)),
                         "train_loss":    float(m.group(2)),
                         "val_quality":   float(m.group(3))})
    return pd.DataFrame(rows)

def parse_stage6(log):
    rows = []
    for line in log.strip().splitlines():
        m = re.match(
            r"Epoch (\d+) \| loss=([\d.]+) \| "
            r"val_q=([\d.]+) val_p=([\d.]+) val_fb_rmse=([\d.]+) \| "
            r"composite=([\d.]+) lr=([\d.e+-]+)",
            line,
        )
        if m:
            rows.append({"epoch":      int(m.group(1)),
                         "train_loss": float(m.group(2)),
                         "val_q":      float(m.group(3)),
                         "val_p":      float(m.group(4)),
                         "val_fb_rmse":float(m.group(5)),
                         "composite":  float(m.group(6)),
                         "lr":         float(m.group(7))})
    return pd.DataFrame(rows)

df3 = parse_stage3(STAGE3_LOG)
df6 = parse_stage6(STAGE6_LOG)
print(f"Stage 3: {len(df3)} epochs parsed")
print(f"Stage 6: {len(df6)} epochs parsed")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training loss
ax1.plot(df3["epoch"], df3["train_loss"], color="#3b6aad", linewidth=2)
ax1.set_xlabel("Epoch");  ax1.set_ylabel("Training loss")
ax1.set_title("Stage 3 — 3DYoga90 Pretrain: training loss")
ax1.grid(alpha=0.3)

# Val quality accuracy with annotation on the saturation
ax2.plot(df3["epoch"], df3["val_quality"], color="#c9552d", linewidth=2,
         marker="o", markersize=4)
ax2.axhline(1.0, linestyle="--", color="gray", alpha=0.6, label="perfect (1.0)")
ax2.axhline(0.5, linestyle=":",  color="gray", alpha=0.4, label="chance (0.5)")
# Highlight the saturation point
sat_epoch = df3[df3["val_quality"] >= 0.999]["epoch"].min()
if pd.notna(sat_epoch):
    ax2.axvline(sat_epoch, linestyle="--", color="#c9552d", alpha=0.4)
    ax2.annotate(f"saturated\nepoch {sat_epoch}",
                 xy=(sat_epoch, 0.999), xytext=(sat_epoch + 2, 0.75),
                 fontsize=9, color="#c9552d",
                 arrowprops=dict(arrowstyle="->", color="#c9552d", alpha=0.6))
ax2.set_xlabel("Epoch");  ax2.set_ylabel("Val quality accuracy")
ax2.set_title("Stage 3 — val quality accuracy (shortcut saturation)")
ax2.set_ylim(0.45, 1.02)
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("path/to/working/results/viz_stage3_training.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_stage3_training.png")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Training loss
ax = axes[0, 0]
ax.plot(df6["epoch"], df6["train_loss"], color="#3b6aad", linewidth=2)
ax.set_xlabel("Epoch");  ax.set_ylabel("Training loss")
ax.set_title("Stage 6 — fine-tune training loss")
ax.grid(alpha=0.3)

# Val quality + val pose accuracy on same axes
ax = axes[0, 1]
ax.plot(df6["epoch"], df6["val_q"], color="#c9552d", linewidth=2,
        marker="o", markersize=3, label="val_quality")
ax.plot(df6["epoch"], df6["val_p"], color="#2e8b57", linewidth=2,
        marker="s", markersize=3, label="val_pose")
ax.axhline(1.0, linestyle="--", color="gray", alpha=0.4)
ax.set_xlabel("Epoch");  ax.set_ylabel("Val accuracy")
ax.set_title("Stage 6 — val accuracies (quality + pose)")
ax.set_ylim(0.55, 1.02)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

# Val feedback RMSE (degrees)
ax = axes[1, 0]
ax.plot(df6["epoch"], df6["val_fb_rmse"], color="#7d4da6", linewidth=2,
        marker="^", markersize=3)
ax.set_xlabel("Epoch");  ax.set_ylabel("Val feedback RMSE (degrees)")
ax.set_title("Stage 6 — val feedback head RMSE")
ax.set_ylim(7.5, 9.0)
ax.grid(alpha=0.3)

# Composite score with LR drops overlaid
ax = axes[1, 1]
ax.plot(df6["epoch"], df6["composite"], color="#444", linewidth=2,
        marker="o", markersize=3, label="composite = q + p")
# Mark LR drops
lr_changes = df6.loc[df6["lr"].diff().fillna(0) < 0, "epoch"].tolist()
for ep in lr_changes:
    ax.axvline(ep, linestyle=":", color="#888", alpha=0.7)
    ax.annotate(f"LR→{df6.loc[df6.epoch==ep, 'lr'].iloc[0]:.0e}",
                xy=(ep, 1.85), xytext=(ep + 0.3, 1.78),
                fontsize=8, color="#666", rotation=90)
# Mark best composite
best_idx = df6["composite"].idxmax()
best_ep, best_comp = df6.loc[best_idx, "epoch"], df6.loc[best_idx, "composite"]
ax.scatter([best_ep], [best_comp], color="gold", edgecolor="black",
           s=120, zorder=5, label=f"best (ep {best_ep}, {best_comp:.4f})")
ax.set_xlabel("Epoch");  ax.set_ylabel("Composite (val_q + val_p)")
ax.set_title("Stage 6 — composite score w/ LR schedule")
ax.set_ylim(1.4, 1.95)
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("path/to/working/results/viz_stage6_training.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_stage6_training.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Three points + dashed connectors
stages = [
    {"label": "Stage 3\n3DYoga90 pretrain\n(on 3DYoga90 test)",
     "pose_acc": 0.9475, "color": "#3b6aad"},
    {"label": "Stage 5\npretrain on YogNet\n(no fine-tune)",
     "pose_acc": 0.4561, "color": "#c9552d"},
    {"label": "Stage 6\nfine-tuned on YogNet\n(on YogNet test)",
     "pose_acc": 0.9631, "color": "#2e8b57"},
]
xs = [0, 1, 2]
accs = [s["pose_acc"] for s in stages]
colors = [s["color"] for s in stages]
ax.plot(xs, accs, linestyle="--", color="gray", alpha=0.5, zorder=1)
ax.scatter(xs, accs, c=colors, s=300, zorder=3, edgecolor="black", linewidth=1.5)
for i, s in enumerate(stages):
    ax.annotate(f"{s['pose_acc']*100:.1f}%",
                xy=(xs[i], accs[i]), xytext=(0, 14),
                textcoords="offset points", ha="center",
                fontsize=13, fontweight="bold", color=s["color"])

ax.set_xticks(xs)
ax.set_xticklabels([s["label"] for s in stages], fontsize=10)
ax.set_ylabel("Pose classification accuracy (7-way)", fontsize=11)
ax.set_title("Pose head — domain shift and recovery across stages",
             fontsize=13)
ax.set_ylim(0.0, 1.05)
ax.axhline(1.0/7, linestyle=":", color="gray", alpha=0.5)
ax.text(2.2, 1.0/7 + 0.02, "random chance (1/7)",
        fontsize=9, color="gray", va="bottom", ha="right")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("path/to/working/results/viz_pose_recovery.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_pose_recovery.png")


In [ ]:
POSE_NAMES = ["cobra", "corpse", "downward_facing_dog",
              "mountain", "tree", "triangle", "warrior"]

# From Stage 3 Cell 6 output
stage3_f1 = {"cobra": 0.952, "corpse": 0.982, "downward_facing_dog": 0.961,
             "mountain": 0.971, "tree": 0.943, "triangle": 0.930, "warrior": 0.893}
# From Stage 6 Cell 7 output
stage6_f1 = {"cobra": 0.975, "corpse": 0.922, "downward_facing_dog": 0.966,
             "mountain": 0.970, "tree": 0.988, "triangle": 0.953, "warrior": 0.959}
# Stage 5 per-class F1 (pretrain on YogNet, no finetune)
stage5_f1 = {"cobra": 0.169, "corpse": 0.303, "downward_facing_dog": 0.697,
             "mountain": 0.284, "tree": 0.557, "triangle": 0.678, "warrior": 0.370}

x = np.arange(len(POSE_NAMES))
width = 0.27

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, [stage3_f1[p] for p in POSE_NAMES], width,
       label="Stage 3 (pretrain, 3DYoga90 test)", color="#3b6aad")
ax.bar(x,         [stage5_f1[p] for p in POSE_NAMES], width,
       label="Stage 5 (pretrain → YogNet, no finetune)", color="#c9552d")
ax.bar(x + width, [stage6_f1[p] for p in POSE_NAMES], width,
       label="Stage 6 (finetuned, YogNet test)", color="#2e8b57")

ax.set_xticks(x)
ax.set_xticklabels([p.replace("_", "\n") for p in POSE_NAMES], fontsize=9)
ax.set_ylabel("F1 score");  ax.set_ylim(0, 1.05)
ax.set_title("Per-pose F1 score across pipeline stages", fontsize=12)
ax.axhline(1.0, linestyle=":", color="gray", alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("path/to/working/results/viz_per_pose_f1.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_per_pose_f1.png")


In [ ]:
JOINT_NAMES = ["left_elbow", "right_elbow", "left_knee",
               "right_knee", "left_hip", "right_hip"]

# Stage 3 test MAE (3DYoga90)
stage3_mae = [3.23, 2.49, 2.72, 3.52, 3.07, 3.39]
# Stage 6 test MAE on CORRUPTED samples (YogNet)
stage6_mae_corrupted = [5.44, 7.68, 4.81, 7.55, 5.71, 5.28]

x = np.arange(len(JOINT_NAMES))
width = 0.38

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - width/2, stage3_mae, width,
       label="Stage 3 — 3DYoga90 test (all windows)",
       color="#3b6aad")
ax.bar(x + width/2, stage6_mae_corrupted, width,
       label="Stage 6 — YogNet test (corrupted samples only)",
       color="#2e8b57")

# Annotate each bar with value
for i, v in enumerate(stage3_mae):
    ax.text(i - width/2, v + 0.15, f"{v:.2f}", ha="center", fontsize=9)
for i, v in enumerate(stage6_mae_corrupted):
    ax.text(i + width/2, v + 0.15, f"{v:.2f}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([j.replace("_", "\n") for j in JOINT_NAMES], fontsize=9)
ax.set_ylabel("Feedback head MAE (degrees)")
ax.set_ylim(0, max(stage6_mae_corrupted) * 1.2)
ax.set_title("Feedback head per-joint MAE — pretrain vs. fine-tune",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="y")

# Annotate the right-side degradation
ax.text(0.5, 0.95,
        "Right-side joints ~1.5× worse on YogNet — MediaPipe occlusion under\n"
        "four-view recording (back/left views partly occlude right limbs)",
        transform=ax.transAxes, fontsize=9, ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff2cc",
                  edgecolor="#c5a900", alpha=0.9))

plt.tight_layout()
plt.savefig("path/to/working/results/viz_feedback_mae.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_feedback_mae.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# From Stage 6 Cell 7 output
cm = np.array([[426, 62],
               [21, 467]])
labels = ["correct", "incorrect"]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())

# Annotate cells
for i in range(2):
    for j in range(2):
        value = cm[i, j]
        pct = value / cm.sum() * 100
        text_color = "white" if value > cm.max() * 0.5 else "black"
        ax.text(j, i, f"{value}\n({pct:.1f}%)",
                ha="center", va="center", color=text_color,
                fontsize=12, fontweight="bold")

ax.set_xticks([0, 1]);  ax.set_yticks([0, 1])
ax.set_xticklabels(labels);  ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Stage 6 — quality head confusion (YogNet test)")

# Recall on each row
for i, lbl in enumerate(labels):
    recall = cm[i, i] / cm[i].sum()
    ax.text(-0.7, i, f"recall\n{recall:.2%}",
            ha="center", va="center", fontsize=9, color="#555")

plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("path/to/working/results/viz_quality_confusion.png", dpi=120,
            bbox_inches="tight")
plt.show()
print("Saved: viz_quality_confusion.png")


In [ ]:
import os, zipfile

RESULTS = "path/to/working/results"
viz_files = [
    "viz_stage3_training.png",
    "viz_stage6_training.png",
    "viz_pose_recovery.png",
    "viz_per_pose_f1.png",
    "viz_feedback_mae.png",
    "viz_quality_confusion.png",
]

# Make sure they landed in results/
for f in viz_files:
    path = os.path.join(RESULTS, f)
    exists = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    print(f"  {'✓' if exists else '✗'} {f}  ({size_kb:.0f} KB)")

# Rebuild the bundle zip
zip_path = "path/to/working/cs7150_yoga_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RESULTS):
        for fn in files:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, RESULTS)
            zf.write(full, f"cs7150_yoga_results/{rel}")

print(f"\nBundle updated: {zip_path}  "
      f"({os.path.getsize(zip_path)/1024/1024:.1f} MB)")


In [ ]:
# 7×7 pose confusion matrix from Stage 6 test set
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Assumes model + test_ds + POSE_NAMES are still in scope from Stage 6.
# If not, reload by rerunning Stage 6 Cells 2 and 5.

test_loader_eval = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
all_p, all_t = [], []
model.eval()
with torch.no_grad():
    for Xb, _, yb_p, _ in test_loader_eval:
        Xb = Xb.to(device)
        _, logits_p, _ = model(Xb)
        all_p.append(logits_p.argmax(1).cpu().numpy())
        all_t.append(yb_p.numpy())
all_p = np.concatenate(all_p)
all_t = np.concatenate(all_t)

cm        = confusion_matrix(all_t, all_p, labels=list(range(len(POSE_NAMES))))
cm_norm   = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalized
short     = [p.replace("downward_facing_dog", "down_dog") for p in POSE_NAMES]

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

for i in range(len(POSE_NAMES)):
    for j in range(len(POSE_NAMES)):
        value = cm[i, j]
        pct   = cm_norm[i, j] * 100
        if value == 0:
            continue
        text_color = "white" if cm_norm[i, j] > 0.5 else "black"
        text = f"{value}" if i != j else f"{value}\n({pct:.0f}%)"
        ax.text(j, i, text, ha="center", va="center",
                color=text_color, fontsize=9,
                fontweight="bold" if i == j else "normal")

ax.set_xticks(range(len(POSE_NAMES)))
ax.set_yticks(range(len(POSE_NAMES)))
ax.set_xticklabels(short, rotation=40, ha="right", fontsize=9)
ax.set_yticklabels(short, fontsize=9)
ax.set_xlabel("Predicted pose", fontsize=11)
ax.set_ylabel("True pose",      fontsize=11)
ax.set_title(f"Stage 6 pose head — 7×7 confusion matrix (YogNet test)\n"
             f"overall accuracy = {(all_p == all_t).mean()*100:.1f}%",
             fontsize=12)

# Per-class recall in a sidebar
for i, pname in enumerate(short):
    recall = cm[i, i] / cm[i].sum()
    ax.text(len(POSE_NAMES) - 0.35, i, f"{recall*100:.0f}%",
            ha="left", va="center", fontsize=9, color="#555")
ax.text(len(POSE_NAMES) - 0.35, -0.7, "recall",
        ha="left", va="center", fontsize=9, color="#555", fontstyle="italic")

plt.colorbar(im, ax=ax, fraction=0.04, label="Row-normalized fraction")
plt.tight_layout()

import os
os.makedirs("path/to/working/results", exist_ok=True)
plt.savefig("path/to/working/results/viz_pose_confusion_7x7.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: viz_pose_confusion_7x7.png")
